# DM-AHR-ViT — standalone training notebook

Every implementation step is a real code cell below, with a markdown heading. The notebook
contains the project's actual code, flattened into one shared namespace: no external repository,
no script files, no embedded source strings, no subprocess calls. It trains the diffusion and ViT
models and stops. No evaluation, XAI, reports, or external validation.

Full run: many GPU-hours. A100 recommended; T4 works but is slower. Data and checkpoints are on Drive so every phase resumes.


## 1. Configuration


In [ ]:
FAST_MODE = False
DIFF_BASE_ITERS = 50000
DIFF_FINETUNE_ITERS = 20000
CV2_STEPS, CV2_START_T, CV2_SHARDS = 50, 600, 4
SEEDS_FULL = [42, 123, 2026, 7, 11, 17, 23, 31]
SEEDS_PRE = [42, 123, 2026]
SEEDS = SEEDS_PRE if FAST_MODE else SEEDS_FULL
ARMS = ["A", "Astar", "B", "Cv2"]

import os, sys, time, json, hashlib, shutil, logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import cv2
import torch
import requests
from PIL import Image

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/dm_ahr_vit")
else:
    BASE = Path.cwd() / "_dm_ahr_run"
BASE.mkdir(parents=True, exist_ok=True)
DATA_ROOT, ARTIFACT_ROOT = BASE / "data", BASE / "artifacts"
for p in (DATA_ROOT, ARTIFACT_ROOT):
    p.mkdir(parents=True, exist_ok=True)
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
REPO_ROOT = BASE
log = logging.getLogger("dm_ahr_vit")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
print("BASE", BASE, "| seeds", SEEDS)


## 2. Paths and configuration loader

The project resolves every path from `DATA_ROOT` and `ARTIFACT_ROOT`. The JSON configs are kept as Python dictionaries here so the notebook needs no files other than the data it downloads.


In [ ]:
def ensure_dirs():
    for p in (data_root(), isic_images_dir(), ham_root(), ham_masks_dir(), manifest_dir(),
              artifact_dir("logs"), artifact_dir("checkpoints"), artifact_dir("predictions"),
              artifact_dir("panels"), artifact_dir("reports"), data_root() / "hair_layers"):
        p.mkdir(parents=True, exist_ok=True)

def data_root(): return Path(os.environ.get("DATA_ROOT", REPO_ROOT / "data"))
def artifact_root(): return Path(os.environ.get("ARTIFACT_ROOT", REPO_ROOT / "artifacts"))
def raw_dir(): return data_root() / "raw"
def isic_images_dir(): return raw_dir() / "isic_212" / "images"
def ham_root(): return raw_dir() / "ham10000"
def ham_masks_dir(): return ham_root() / "masks"
def manifest_dir(): return data_root() / "manifests"
def processed_dir(arm): return data_root() / "processed" / arm
def artifact_dir(*parts):
    p = artifact_root().joinpath(*parts); p.mkdir(parents=True, exist_ok=True); return p

CONFIGS = {'dmahr.json': {'version': '1.0.0', 'upstream_repo': 'https://github.com/AnasHXH/DM-AHR', 'upstream_commit': '0aee1bbabf788a59be5e1eecddc20f533b5006d8', 'native_resolution': 128, 'condition_resolution': 128, 'num_timesteps': 2000, 'beta_schedule': {'schedule': 'linear', 'linear_start': 1e-06, 'linear_end': 0.01}, 'unet': {'in_channel': 6, 'out_channel': 3, 'inner_channel': 64, 'channel_multiplier': [1, 2, 4, 8, 8], 'attn_res': [16], 'res_blocks': 2, 'dropout': 0.2}, 'train': {'iterations': 20000, 'batch_size': 4, 'lr': 0.0001, 'optimizer': 'adam', 'checkpoint_every': 1000, 'val_every': 1000, 'snapshot_every': 5000, 'overfit_pairs': 12}, 'sampling': {'batch_size': 32, 'num_timesteps': 2000, 'reduced_steps_ablation': None}}, 'dullrazor.json': {'config_hash': '0af616f50f77e1c57c4fff1e1f6bb8c2ab99100052c703d937bccbbf109a9d70', 'frozen': True, 'inpaint_method': 'telea', 'inpaint_radius': 3, 'mask_dilate_iterations': 1, 'mask_dilate_kernel': 3, 'version': '1.0.0'}, 'hair_screen.json': {'blackhat_threshold': 18, 'config_hash': 'e232efe4705133a175ec50d5147954a6545456d469b6353fb42d348387704273', 'coverage_threshold': 0.01, 'dilate_iterations': 2, 'dilate_kernel': 5, 'frozen': True, 'kernel_length': 17, 'light_hair': True, 'min_component_area': 40, 'orientations': 12, 'tophat_threshold': 22, 'version': '1.0.0'}, 'protocol.json': {'study': 'DM-AHR-ViT', 'version': '1.0.0', 'max_spend_usd': 250.0, 'seeds': [42, 123, 2026], 'split_seed': 42, 'split_fractions': {'train': 0.7, 'val': 0.15, 'test': 0.15}, 'classes': ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'], 'image_size': 224, 'isic_collection_id': 212, 'expected_c0_total': 10015, 'expected_collection_total': 11720, 'near_duplicate_hamming': 6, 'arms': ['A', 'B', 'C', 'Astar'], 'primary_endpoints': ['test_macro_f1', 'paired_psnr', 'paired_ssim', 'x1_gradcam_overlap20', 'x1_rollout_overlap20'], 'planned_comparisons': [['A', 'B'], ['A', 'C'], ['B', 'C']], 'bootstrap_replicates': 10000, 'bootstrap_seed': 20260101, 'research_only_statement': 'Research prototype only. Not clinically validated and not intended for diagnosis or patient care.'}, 'sampling.json': {'version': '1.0.0', 'psnr_data_range': 1.0, 'ssim_data_range': 1.0, 'ssim_win_size': 7, 'xai_top_fraction': 0.2, 'xai_pointing_tolerance_px': 0, 'chance_baselines': ['uniform', 'centered_gaussian']}, 'stale_phrases.json': {'comment': 'Phrases that must not appear in the current deliverables. Each pattern is a case-insensitive regex. History belongs in decision_log.md, which is not scanned.', 'files': ['reports/client_summary.md', 'reports/final_research_report.md', 'reports/stageE_results.md', 'reports/stageC_results.md', 'reports/xai_corrected_results.md'], 'docx_dir': 'artifacts/reports', 'patterns': ['no arm beats a centre prior', 'below the centred-Gaussian baseline', 'centred[- ]Gaussian baseline[^.\\n]{0,20}0\\.64', 'share of the top 20% attention inside the lesion', 'top 20% most salient pixels that fall inside the mask', '\\bd_?z\\b', 'effect_size_dz', 'over-?smooth(ed|ing)? (output|images|diffusion)', 'Grad-CAM[^.\\n]{0,60}(visual )?faithfulness has been validated', 'attributable to the hair-removal condition alone', 'classical removal hurt it', 'drives both its quality scores and its classification deficit', 'only manipulated variable', 'larger share of the 128 px'], 'paper_files': ['artifacts/reports/Research_Paper.docx'], 'paper_patterns': ['\\bColab\\b', '\\bthe client\\b', 'expected results', '(is|are) expected to (achieve|show|produce|preserve)', 'rotations? of up to [±+]/?-?\\s?90', 'improves not only classification', '(?<!knowledge, )no prior study has', 'to be agreed by']}, 'vit.json': {'version': '1.0.0', 'model': 'vit_b_16', 'weights': 'IMAGENET1K_V1', 'image_size': 224, 'batch_size': 32, 'grad_accum_steps': 1, 'epochs': 100, 'lr': 3e-05, 'weight_decay': 0.05, 'warmup_epochs': 5, 'scheduler': 'cosine', 'label_smoothing': 0.0, 'class_weights': 'train_inverse_frequency', 'early_stopping_patience': 10, 'early_stopping_metric': 'val_macro_f1', 'grad_clip': 1.0, 'precision': 'bf16', 'num_workers': 8, 'augmentation': {'random_resized_crop_scale': [0.75, 1.0], 'random_resized_crop_ratio': [0.9, 1.1111111111111112], 'horizontal_flip': 0.5, 'vertical_flip': 0.5, 'rotation_degrees': 15, 'color_jitter': [0.15, 0.15, 0.1, 0.02]}}, 'akiec.json': {'name': 'sr_ffhq', 'phase': 'train', 'gpu_ids': [0], 'path': {'log': 'logs', 'tb_logger': 'tb_logger', 'results': 'results', 'checkpoint': 'checkpoint', 'resume_state': 'experiments/sr_ffhq_231220_121203/checkpoint/I725000_E1083'}, 'datasets': {'train': {'name': 'FFHQ', 'mode': 'HR', 'dataroot': 'hair_removal/akiec_16_128', 'datatype': 'img', 'l_resolution': 16, 'r_resolution': 128, 'batch_size': 4, 'num_workers': 1, 'use_shuffle': True, 'data_len': -1}, 'val': {'name': 'CelebaHQ', 'mode': 'LRHR', 'dataroot': 'hair_removal/akiec_16_128', 'datatype': 'img', 'l_resolution': 16, 'r_resolution': 128, 'data_len': -1}}, 'model': {'which_model_G': 'sr3', 'finetune_norm': False, 'unet': {'in_channel': 6, 'out_channel': 3, 'inner_channel': 64, 'channel_multiplier': [1, 2, 4, 8, 8], 'attn_res': [16], 'res_blocks': 2, 'dropout': 0.2}, 'beta_schedule': {'train': {'schedule': 'linear', 'n_timestep': 2000, 'linear_start': 1e-06, 'linear_end': 0.01}, 'val': {'schedule': 'linear', 'n_timestep': 2000, 'linear_start': 1e-06, 'linear_end': 0.01}}, 'diffusion': {'image_size': 128, 'channels': 3, 'conditional': True}}, 'train': {'n_iter': 660000, 'val_freq': 10000.0, 'save_checkpoint_freq': 10000.0, 'print_freq': 200, 'optimizer': {'type': 'adam', 'lr': 0.0001}, 'ema_scheduler': {'step_start_ema': 5000, 'update_ema_every': 1, 'ema_decay': 0.9999}}, 'wandb': {'project': 'sr_ffhq'}}}
def load_config(path):
    key = str(path).split("/")[-1]
    if key not in CONFIGS:
        raise KeyError(f"config {key} not embedded")
    return json.loads(json.dumps(CONFIGS[key]))
def load_frozen(name, default): return load_config(default)
def config_hash(cfg): return hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()
def config_hash_file(path): return config_hash(load_config(path))
def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, default=str))
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""): h.update(c)
    return h.hexdigest()
def timed(name, lg):
    class T:
        def __enter__(self): self.t = time.time(); return self
        def __exit__(self, *a): self["seconds"] = round(time.time() - self.t, 1)
        def __getitem__(self, k): return getattr(self, k)
    return T()
def record_run(tag, **kw): pass
def load_hair_screen_config():
    cfg = load_config("hair_screen.json")
    h = cfg.get("config_hash")
    if h and h != config_hash({k: v for k, v in cfg.items() if k != "config_hash"}):
        raise RuntimeError("hair screen config hash mismatch")
    return cfg
CLASSES = ("akiec", "bcc", "bkl", "df", "mel", "nv", "vasc")
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
EXPECTED_C0_COUNTS = {"akiec": 327, "bcc": 514, "bkl": 1099, "df": 115,
                      "mel": 1113, "nv": 6705, "vasc": 142}
EXPECTED_C0_TOTAL = 10015
def setup_logging(): return logging.getLogger("dm_ahr_vit")
ensure_dirs()
print("configs:", list(CONFIGS))

# No-op redefinitions remove the last flattening stubs.
def _ensure_path():
    return None


## 3a. ISIC collection download

Fetches image metadata from the public API and the JPEGs from S3, resumably.


In [ ]:
"""ISIC Archive collection download."""


import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests


def download_collection(collection_id: int, images_dir: Path, records_path: Path,
                        workers: int = 16, limit: int | None = None) -> dict:
    """Download one ISIC collection, recording per-file status to records.jsonl."""
    images_dir.mkdir(parents=True, exist_ok=True)
    records_path.parent.mkdir(parents=True, exist_ok=True)
    api = f"https://api.isic-archive.com/api/v2/images?collection_id={collection_id}"
    images, page = [], 0
    while True:
        r = requests.get(api, params={"limit": 100, "offset": page * 100}, timeout=60)
        r.raise_for_status()
        batch = r.json().get("results", [])
        if not batch:
            break
        images.extend(batch)
        page += 1
        if limit and len(images) >= limit:
            images = images[:limit]
            break

    def fetch(img):
        iid = img["isic_id"]
        out = images_dir / f"{iid}.jpg"
        url = f"https://isic-archive.s3.amazonaws.com/images/{iid}.jpg"
        rec = {"isic_id": iid, "url": url, "bytes": None, "sha256": None,
               "status": "failed", "attribution": img.get("attribution"),
               "license": img.get("license"),
               "expected_size": img.get("metadata", {}).get("acquisition", {}).get("pixels")}
        try:
            if out.exists():
                blob = out.read_bytes()
            else:
                resp = requests.get(url, timeout=120)
                resp.raise_for_status()
                blob = resp.content
                out.write_bytes(blob)
            rec.update({"bytes": len(blob), "sha256": hashlib.sha256(blob).hexdigest(),
                        "status": "downloaded"})
        except Exception as exc:
            rec["status"] = f"failed: {exc}"
        return rec

    with ThreadPoolExecutor(max_workers=workers) as ex:
        records = list(ex.map(fetch, images))
    with records_path.open("w") as fh:
        for rec in records:
            fh.write(json.dumps(rec) + "\n")
    return {"collection": collection_id, "n": len(records),
            "downloaded": sum(r["status"] == "downloaded" for r in records)}


## 3b. HAM10000 metadata and lesion masks

Harvard Dataverse metadata plus the released lesion segmentations.


In [ ]:
"""HAM10000 metadata (Harvard Dataverse) and cohort helpers."""


import zipfile
from pathlib import Path

import pandas as pd


DATAVERSE_DOI = "10.7910/DVN/DBW86T"
DATAVERSE_ACCESS = "https://dataverse.harvard.edu/api/access/datafile/{fid}"
METADATA_FILE_ID = 3282766
MASKS_FILE_ID = 3282767


def read_metadata(path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df["image_id"] = df["image_id"].astype(str)
    df["lesion_id"] = df["lesion_id"].astype(str)
    return df


def class_counts(df: pd.DataFrame) -> dict:
    counts = df["dx"].value_counts().to_dict()
    return {c: int(counts.get(c, 0)) for c in CLASSES}


def expected_comparison(metadata: pd.DataFrame) -> dict:
    counts = class_counts(metadata)
    return {
        "recomputed": counts,
        "expected": EXPECTED_C0_COUNTS,
        "total": int(sum(counts.values())),
        "expected_total": EXPECTED_C0_TOTAL,
        "matches": all(counts[c] == EXPECTED_C0_COUNTS[c] for c in CLASSES),
    }


def extract_masks(zip_path: Path, out_dir: Path) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith("/"):
                continue
            target = out_dir / Path(name).name
            if not target.exists():
                target.write_bytes(zf.read(name))
            n += 1
    return {"extracted": n, "out_dir": str(out_dir)}


## 3c. Manifest, perceptual hash, near-duplicate confirmation

Builds the image manifest, 64-bit dHash pairs and pixel-confirmed duplicate links.


In [ ]:
"""File manifests: decode audit, dHash near-duplicate candidates, confirmation."""


import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image


def _describe(path: Path, image_id: str) -> dict:
    row = {
        "image_id": image_id, "decode_status": "fail_missing",
        "width": None, "height": None, "mode": None, "format": None,
        "sha256": None, "bytes": None,
    }
    if not path.exists():
        return row
    row["bytes"] = path.stat().st_size
    try:
        with Image.open(path) as fh:
            fh.load()
            w, h = fh.size
            row.update({"decode_status": "ok", "width": w, "height": h,
                        "mode": fh.mode, "format": fh.format})
    except Exception:
        row["decode_status"] = "fail_decode"
        return row
    row["sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()
    return row


def dhash(img: Image.Image, size: int = 8) -> int:
    """64-bit difference hash on a (size+1)x(size) grayscale grid."""
    g = np.asarray(img.convert("L").resize((size + 1, size), Image.Resampling.BILINEAR),
                   dtype=np.int16)
    bits = g[:, 1:] > g[:, :-1]
    v = 0
    for b in bits.ravel():
        v = (v << 1) | int(b)
    return v


def hamming_pairs(hashes: np.ndarray, threshold: int, ids: list) -> list:
    n = len(hashes)
    out = []
    for i in range(n):
        for j in range(i + 1, n):
            d = bin(int(hashes[i]) ^ int(hashes[j])).count("1")
            if d <= threshold:
                out.append({"id_a": ids[i], "id_b": ids[j], "distance": d})
    return out


def confirm_near_duplicates(candidates: pd.DataFrame, man: pd.DataFrame,
                            max_mean_abs_diff: float = 4.0,
                            min_correlation: float = 0.985) -> pd.DataFrame:
    """Pixel-level confirmation of dHash candidates: mean abs diff and correlation."""
    import cv2

    paths = dict(zip(man["image_id"], man["path"]))
    rows = []
    for r in candidates.itertuples(index=False):
        try:
            a = cv2.imread(str(Path(paths[r.id_a])), cv2.IMREAD_GRAYSCALE)
            b = cv2.imread(str(Path(paths[r.id_b])), cv2.IMREAD_GRAYSCALE)
            if a is None or b is None or a.shape != b.shape:
                rows.append({"id_a": r.id_a, "id_b": r.id_b, "dhash_distance": r.distance,
                             "mean_abs_diff": float("inf"), "correlation": 0.0,
                             "confirmed": False})
                continue
            mad = float(np.abs(a.astype(np.float64) - b.astype(np.float64)).mean())
            va, vb = a.astype(np.float64).ravel(), b.astype(np.float64).ravel()
            corr = float(np.corrcoef(va, vb)[0, 1]) if va.std() > 0 and vb.std() > 0 else 0.0
            rows.append({"id_a": r.id_a, "id_b": r.id_b, "dhash_distance": r.distance,
                         "mean_abs_diff": mad, "correlation": corr,
                         "confirmed": bool(mad <= max_mean_abs_diff and corr >= min_correlation)})
        except Exception:
            rows.append({"id_a": r.id_a, "id_b": r.id_b, "dhash_distance": r.distance,
                         "mean_abs_diff": float("inf"), "correlation": 0.0, "confirmed": False})
    return pd.DataFrame(rows)


def build_manifest(records_path: Path, images_dir: Path, metadata: pd.DataFrame,
                   out_dir: Path):
    """Per-image manifest from the download records + HAM metadata."""
    records = [json.loads(l) for l in Path(records_path).read_text().splitlines() if l.strip()]
    meta = metadata.set_index("image_id")
    rows, excl = [], []
    for rec in records:
        iid = rec["isic_id"]
        path = images_dir / f"{iid}.jpg"
        d = _describe(path, iid)
        d.update({
            "path": str(path),
            "in_c0": iid in meta.index,
            "cohort": "C0" if iid in meta.index else "C0-extra",
            "license": rec.get("license"),
            "attribution": rec.get("attribution"),
            "expected_bytes": rec.get("expected_size"),
            "download_status": rec.get("status"),
        })
        if iid in meta.index:
            d["dx"] = str(meta.loc[iid, "dx"])
            d["lesion_id"] = str(meta.loc[iid, "lesion_id"])
        else:
            d["dx"] = None
            d["lesion_id"] = None
            excl.append({"image_id": iid, "cohort": "C0-extra",
                         "reason": "not in HAM10000 metadata", "path": str(path)})
        if d["decode_status"] != "ok":
            excl.append({"image_id": iid, "cohort": d["cohort"],
                         "reason": d["decode_status"], "path": str(path)})
        if d["decode_status"] == "ok":
            with Image.open(path) as fh:
                d["dhash"] = dhash(fh)
        else:
            d["dhash"] = None
        rows.append(d)
    df = pd.DataFrame(rows)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / "data_manifest.csv", index=False)
    pd.DataFrame(excl).to_csv(out_dir / "exclusions.csv", index=False)
    return df, pd.DataFrame(excl), {"n": len(df)}


## 3d. Lesion-grouped 70/15/15 split

Groups by lesion plus confirmed duplicates, stratifies, freezes and verifies zero leakage.


In [ ]:
"""Lesion-grouped splits: union of lesion_id and confirmed near-duplicates."""


import hashlib
from pathlib import Path

import numpy as np
import pandas as pd



class UnionFind:
    def __init__(self):
        self.parent: dict = {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
            return x
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:  # path compression
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


def build_groups(c0: pd.DataFrame, near: pd.DataFrame) -> pd.DataFrame:
    """Merge images sharing a lesion_id or a confirmed near-duplicate link."""
    uf = UnionFind()
    for lesion in c0["lesion_id"].unique():
        members = c0.loc[c0["lesion_id"] == lesion, "image_id"].tolist()
        for m in members[1:]:
            uf.union(members[0], m)
    if len(near):
        for r in near.itertuples(index=False):
            uf.union(r.id_a, r.id_b)
    df = c0.copy()
    first_seen: dict = {}
    group_ids = []
    for iid in df["image_id"]:
        root = uf.find(iid)
        if root not in first_seen:
            first_seen[root] = f"G{len(first_seen):05d}"
        group_ids.append(first_seen[root])
    df["group_id"] = group_ids
    return df


def freeze_splits(grouped: pd.DataFrame, seed: int, fractions: dict,
                  out_dir, protocol_hash: str) -> dict:
    """Stratified group split, frozen to CSV plus its SHA-256."""
    rng = np.random.default_rng(seed)
    order = ["train", "val", "test"]
    fr = [fractions[s] for s in order]
    assign: dict = {}
    for dx in CLASSES:
        groups = sorted(grouped.loc[grouped["dx"] == dx, "group_id"].unique())
        rng.shuffle(groups)
        n = len(groups)
        n_tr = int(round(fr[0] * n))
        n_va = int(round(fr[1] * n))
        for g in groups[:n_tr]:
            assign[g] = "train"
        for g in groups[n_tr:n_tr + n_va]:
            assign[g] = "val"
        for g in groups[n_tr + n_va:]:
            assign[g] = "test"
    df = grouped.copy()
    df["split"] = df["group_id"].map(assign)
    df = df[["image_id", "lesion_id", "group_id", "dx", "split", "sha256"]]
    df = df.sort_values("image_id").reset_index(drop=True)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / "splits.csv", index=False)
    digest = hashlib.sha256((out_dir / "splits.csv").read_bytes()).hexdigest()
    (out_dir / "splits.sha256").write_text(digest + "\n")
    counts = {s: df[df["split"] == s]["dx"].value_counts().to_dict() for s in order}
    return {
        "counts": counts,
        "n_groups": int(df["group_id"].nunique()),
        "splits_sha256": digest,
        "protocol_hash": protocol_hash,
    }


def verify_splits(df: pd.DataFrame) -> dict:
    """No group may appear in more than one split."""
    problems = []
    splits = ["train", "val", "test"]
    for i, a in enumerate(splits):
        for b in splits[i + 1:]:
            ga = set(df.loc[df["split"] == a, "group_id"])
            gb = set(df.loc[df["split"] == b, "group_id"])
            if ga & gb:
                problems.append(f"group overlap between {a} and {b}: {len(ga & gb)} groups")
    if df["image_id"].duplicated().any():
        problems.append("duplicate image_id in split frame")
    sizes = {s: int((df["split"] == s).sum()) for s in splits}
    return {"ok": not problems, "problems": problems, "sizes": sizes}


## 3e. Lesion-mask manifest

Checks the released masks against images and records QA status.


In [ ]:
"""Lesion-mask loading (X1 tier) and the mask manifest builder."""


from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image


X1_PROVENANCE = (
    "HAM10000_segmentations_lesion_tschandl (Harvard Dataverse doi:10.7910/DVN/DBW86T)"
)


def resolve(path) -> Path:
    """Re-root container-recorded ``/__modal/...`` paths onto the local data root."""
    p = Path(path)
    if p.exists():
        return p
    marker = "/data/"
    if marker in str(p):
        return data_root() / str(p).split(marker, 1)[1]
    return p


def load_mask_224(path, size: int = 224) -> np.ndarray:
    """Grayscale mask -> binary {0,1} array resized with NEAREST."""
    with Image.open(resolve(path)) as fh:
        m = fh.convert("L").resize((size, size), Image.Resampling.NEAREST)
    return (np.array(m) > 127).astype(np.uint8)


def load_mask_registered(path, image_size_wh=None, size: int = 224) -> np.ndarray:
    """Lesion mask in the model's frame; delegates to `mask_to_model_frame`."""
    with Image.open(resolve(path)) as fh:
        return mask_to_model_frame(fh, image_size_wh, size)

    with Image.open(resolve(path)) as fh:
        return mask_to_model_frame(fh, image_size_wh, size)


def build_x1_manifest(masks_dir: Path, split_df: pd.DataFrame, man: pd.DataFrame,
                      out_path: Path) -> pd.DataFrame:
    """One row per split image: mask presence, geometry checks, foreground fraction."""
    sizes = dict(zip(man["image_id"], zip(man["width"], man["height"])))
    dx = dict(zip(split_df["image_id"], split_df["dx"]))
    group = dict(zip(split_df["image_id"], split_df["group_id"]))
    split = dict(zip(split_df["image_id"], split_df["split"]))
    rows = []
    for image_id in split_df["image_id"]:
        p = masks_dir / f"{image_id}_segmentation.png"
        row = {
            "image_id": image_id,
            "tier": "X1",
            "split": split[image_id],
            "dx": dx[image_id],
            "group_id": group[image_id],
            "has_mask": bool(p.exists()),
            "mask_path": str(p),
            "provenance": X1_PROVENANCE,
            "expert_traced_claimed": False,
        }
        status = "ok"
        if p.exists():
            with Image.open(p) as fh:
                arr = np.array(fh.convert("L"))
            w, h = arr.shape[1], arr.shape[0]
            binary = set(np.unique(arr)) <= {0, 255}
            non_empty = bool((arr > 127).any())
            fg = float((arr > 127).mean())
            iw, ih = sizes.get(image_id, (None, None))
            size_matches = bool(iw == w and ih == h)
            row.update({
                "width": w, "height": h, "binary": bool(binary),
                "non_empty": non_empty, "foreground_fraction": fg,
                "size_matches": size_matches,
            })
            if not (binary and non_empty and size_matches):
                status = "degraded"
        else:
            row.update({"width": None, "height": None, "binary": None,
                        "non_empty": None, "foreground_fraction": None,
                        "size_matches": None})
            status = "missing_mask"
        row["status"] = status
        rows.append(row)
    df = pd.DataFrame(rows)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    return df


## 4a. Hair detector

Multi-orientation black-hat/top-hat detector with train/val threshold tuning.


In [ ]:
"""Multi-orientation black-hat/top-hat hair detector.

Parameters are tuned on train/validation synthetic pairs only and frozen before
any test image is processed.
"""


import cv2
import numpy as np


def _oriented_kernel(length: int, angle_deg: float) -> np.ndarray:
    kernel = np.zeros((length, length), dtype=np.uint8)
    c = length // 2
    rad = np.deg2rad(angle_deg)
    dx, dy = int(round(c * np.cos(rad))), int(round(c * np.sin(rad)))
    cv2.line(kernel, (c - dx, c - dy), (c + dx, c + dy), 1, 1)
    return kernel


def hair_response(gray: np.ndarray, cfg: dict) -> tuple[np.ndarray, np.ndarray]:
    dark = np.zeros_like(gray)
    light = np.zeros_like(gray)
    for i in range(int(cfg["orientations"])):
        angle = 180.0 * i / int(cfg["orientations"])
        k = _oriented_kernel(int(cfg["kernel_length"]), angle)
        dark = np.maximum(dark, cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k))
        if cfg.get("light_hair", True):
            light = np.maximum(light, cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, k))
    return dark, light


def detect_hair(image_rgb: np.ndarray, cfg: dict) -> tuple[np.ndarray, dict]:
    """Return a binary hair mask and detection statistics."""
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    dark, light = hair_response(gray, cfg)
    mask = (dark > int(cfg["blackhat_threshold"])).astype(np.uint8)
    if cfg.get("light_hair", True):
        mask |= (light > int(cfg["tophat_threshold"])).astype(np.uint8)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    keep = np.zeros_like(mask)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= int(cfg["min_component_area"]):
            keep[labels == i] = 1
    k = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (int(cfg["dilate_kernel"]), int(cfg["dilate_kernel"]))
    )
    keep = cv2.dilate(keep, k, iterations=int(cfg["dilate_iterations"]))
    coverage = float(keep.mean())
    stats_out = {
        "coverage": coverage,
        "hair_present": bool(coverage >= float(cfg["coverage_threshold"])),
        "dark_max": int(dark.max()),
        "light_max": int(light.max()),
    }
    return keep, stats_out


def tune_coverage_threshold(
    pairs: list[tuple[np.ndarray, int]], cfg: dict, grid: np.ndarray | None = None
) -> dict:
    """Pick the coverage threshold maximizing balanced accuracy on labelled pairs.

    `pairs` is [(image_rgb, is_hairy)] from train/validation synthetic pairs.
    """
    grid = grid if grid is not None else np.round(np.arange(0.002, 0.081, 0.002), 4)
    coverages = np.array([detect_hair(img, cfg)[1]["coverage"] for img, _ in pairs])
    labels = np.array([lab for _, lab in pairs])
    best = None
    for thr in grid:
        pred = (coverages >= thr).astype(int)
        tpr = float((pred[labels == 1] == 1).mean()) if (labels == 1).any() else 0.0
        tnr = float((pred[labels == 0] == 0).mean()) if (labels == 0).any() else 0.0
        score = 0.5 * (tpr + tnr)
        if best is None or score > best["balanced_accuracy"]:
            best = {
                "coverage_threshold": float(thr),
                "balanced_accuracy": score,
                "sensitivity": tpr,
                "specificity": tnr,
            }
    return best


## 4b. DullRazor classical removal

Morphological hair mask plus Telea inpainting (Arm B).


In [ ]:
"""DullRazor-style morphology + Telea inpainting baseline (Arm B)."""


import cv2
import numpy as np


def apply_dullrazor(image_rgb: np.ndarray, hair_mask: np.ndarray, cfg: dict) -> np.ndarray:
    mask = (hair_mask > 0).astype(np.uint8) * 255
    if cfg.get("mask_dilate_kernel", 0):
        k = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (int(cfg["mask_dilate_kernel"]), int(cfg["mask_dilate_kernel"])),
        )
        mask = cv2.dilate(mask, k, iterations=int(cfg.get("mask_dilate_iterations", 1)))
    method = cv2.INPAINT_TELEA if cfg.get("inpaint_method", "telea") == "telea" else cv2.INPAINT_NS
    bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    out = cv2.inpaint(bgr, mask, float(cfg.get("inpaint_radius", 3)), method)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)


## 4c. Synthetic hair compositor

Renders hair strands onto clean images for the diffusion training pairs.


In [ ]:
"""Seeded synthetic hair rendering for paired hair-removal evaluation."""


import hashlib

import numpy as np
from PIL import Image


def _rng(image_id: str, seed: int) -> np.random.Generator:
    h = hashlib.sha256(f"{image_id}:{seed}".encode()).digest()
    return np.random.default_rng(int.from_bytes(h[:8], "little"))


def render_hair(img: Image.Image, image_id: str, seed: int = 0):
    """Draw random dark curved hairs; return (hairy image, binary mask, params)."""
    rng = _rng(image_id, seed)
    w, h = img.size
    n_hairs = int(rng.integers(8, 20))
    blur_sigma = float(rng.uniform(0.5, 1.5))
    mask = np.zeros((h, w), dtype=np.uint8)
    for _ in range(n_hairs):
        x0, y0 = rng.uniform(0, w), rng.uniform(0, h)
        angle = rng.uniform(0, 2 * np.pi)
        length = rng.uniform(0.4, 1.2) * max(w, h)
        curve = rng.uniform(-0.6, 0.6)
        thickness = int(rng.integers(1, 4))
        steps = 48
        xs = np.linspace(0, length, steps)
        px = x0 + xs * np.cos(angle)
        py = y0 + xs * np.sin(angle)
        perp = (np.linspace(0, 1, steps) ** 2) * curve * 40.0
        px = px + perp * np.sin(angle + np.pi / 2)
        py = py + perp * np.cos(angle + np.pi / 2)
        for x, y in zip(px, py):
            xi, yi = int(round(x)), int(round(y))
            if 0 <= xi < w and 0 <= yi < h:
                mask[max(0, yi - thickness // 2):yi + thickness // 2 + 1,
                     max(0, xi - thickness // 2):xi + thickness // 2 + 1] = 255
    arr = np.asarray(img.convert("RGB"), dtype=np.float64)
    m = mask > 0
    k = max(1, int(round(blur_sigma)))
    if k > 1:
        from scipy.ndimage import uniform_filter
        soft = uniform_filter(mask.astype(np.float64), size=k)
        m = soft > 0.05
    darkness = rng.uniform(0.25, 0.55)
    out = arr.copy()
    out[m] = arr[m] * darkness
    hairy = Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))
    return hairy, Image.fromarray(np.where(mask > 0, 255, 0).astype(np.uint8)), {
        "n_hairs": n_hairs, "blur_sigma": blur_sigma, "seed": seed, "source": "synthetic",
    }


## 4d. Real-strand harvest

Extracts long thin dark strands from train/val images for the fine-tune.


In [ ]:
"""Harvest real hair from dermoscopic images as thin-structure masks.

The frozen detector (`screening.py`) is built to flag *whether* hair is present and is
deliberately generous: its mask is dilated and picks up lesion texture. For compositing real
hair onto clean images we need the opposite: only long, thin, dark strands.

A dark thin component is kept as hair only when all of these hold:
  * strong response (well above the screening threshold), so faint texture is excluded;
  * long skeleton and small mean width (strands, not blobs or short dashes);
  * few junctions per unit length (crossing strands yes, the dense mesh of pigment network no);
  * not blue or purple (pen ink) and not hugging the image border (dermoscope frame edge).
"""


import cv2
import numpy as np
from skimage.morphology import skeletonize


HARVEST_THRESHOLD = 30
MIN_PEAK_RESPONSE = 80
MIN_COMPONENT_AREA = 40
MIN_SKELETON_LENGTH = 40
MAX_MEAN_WIDTH = 7.0
MAX_BRANCH_DENSITY = 0.03
MAX_BLUE_OVER_RED = 8.0
MAX_BORDER_FRACTION = 0.5
BORDER_PX = 4
MAX_FRACTION = 0.25


def thin_hair_mask(image_rgb: np.ndarray, cfg: dict,
                   min_skeleton: int = MIN_SKELETON_LENGTH,
                   max_width: float = MAX_MEAN_WIDTH,
                   threshold: int = HARVEST_THRESHOLD,
                   min_peak: int = MIN_PEAK_RESPONSE,
                   max_branch_density: float = MAX_BRANCH_DENSITY) -> tuple[np.ndarray, dict]:
    """Return a uint8 {0,1} mask of accepted strands and a stats dict."""
    h, w = image_rgb.shape[:2]
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    dark, _ = hair_response(gray, {**cfg, "light_hair": False})
    mask = (dark > int(threshold)).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    skel = skeletonize(mask > 0)
    skel_u8 = skel.astype(np.uint8)
    skel_len = np.bincount(labels[skel], minlength=n).astype(np.float64)
    nb = cv2.filter2D(skel_u8, -1, np.ones((3, 3), np.float32), borderType=cv2.BORDER_CONSTANT)
    junction = (skel & ((nb - skel_u8) >= 3)).astype(np.uint8)
    # Count branch *points* (clusters of junction pixels), not junction pixels.
    n_j, jlab = cv2.connectedComponents(junction, connectivity=8)
    branches = np.zeros(n, dtype=np.float64)
    if n_j > 1:
        ys, xs = np.nonzero(junction)
        _, first = np.unique(jlab[ys, xs], return_index=True)
        branches = np.bincount(labels[ys[first], xs[first]], minlength=n).astype(np.float64)
    area = stats[:, cv2.CC_STAT_AREA].astype(np.float64)
    peak = np.zeros(n, dtype=np.float64)
    np.maximum.at(peak, labels.ravel(), dark.ravel().astype(np.float64))
    flat = labels.ravel()
    mean_rgb = [np.bincount(flat, weights=image_rgb[..., c].ravel().astype(np.float64),
                            minlength=n) / np.maximum(area, 1) for c in range(3)]
    border = np.zeros((h, w), bool)
    border[:BORDER_PX], border[-BORDER_PX:] = True, True
    border[:, :BORDER_PX], border[:, -BORDER_PX:] = True, True
    border_frac = np.bincount(labels[border], minlength=n) / np.maximum(area, 1)

    keep = np.zeros(n, dtype=bool)
    reasons = {"small": 0, "short": 0, "wide": 0, "weak": 0, "mesh": 0, "ink": 0, "border": 0}
    for i in range(1, n):
        if area[i] < MIN_COMPONENT_AREA:
            reasons["small"] += 1
        elif skel_len[i] < min_skeleton:
            reasons["short"] += 1
        elif area[i] / skel_len[i] > max_width:
            reasons["wide"] += 1
        elif peak[i] < min_peak:
            reasons["weak"] += 1
        elif branches[i] / skel_len[i] > max_branch_density:
            reasons["mesh"] += 1
        elif mean_rgb[2][i] - mean_rgb[0][i] > MAX_BLUE_OVER_RED:
            reasons["ink"] += 1
        elif border_frac[i] > MAX_BORDER_FRACTION:
            reasons["border"] += 1
        else:
            keep[i] = True
    out = keep[labels].astype(np.uint8)
    fraction = float(out.mean())
    info = {"components_total": int(n - 1), "components_kept": int(keep.sum()),
            "hair_pixels": int(out.sum()), "fraction": fraction, "rejected": reasons}
    if fraction > MAX_FRACTION:
        out = np.zeros_like(out)
        info.update(components_kept=0, hair_pixels=0, fraction=0.0)
    return out, info


## 4e. Real-strand composite dataset

Builds train/val composite datasets from harvested strands; test split is never read.


In [ ]:
"""Training pairs made by compositing REAL hair onto clean dermoscopic images.

A clean-ish target image (very low detector hair coverage) gets real hair strands, harvested
by `hair/harvest.py` from a different hairy image, pasted on at full resolution. The pair
(target, composite) is then resized to the diffusion model's native size with the same
bicubic resize the rest of the pipeline uses. Targets and sources always come from the same
split, and only the train and validation splits are accepted: the test split cannot be used.
"""


import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset


CLEAN_MAX_COVERAGE = 0.0167
HAIR_MIN_COVERAGE = 0.03
# Layers with very little harvested hair were where review found ink, bubble rims and lesion dots.
MIN_LAYER_FRACTION = 0.004
ALLOWED_SPLITS = ("train", "val")


def select_pools(screening: pd.DataFrame, split: str) -> tuple[list[str], list[str]]:
    """Target ids (clean-ish) and candidate source ids (clearly hairy) for one split."""
    if split not in ALLOWED_SPLITS:
        raise ValueError(f"split {split!r} is not allowed; only {ALLOWED_SPLITS}")
    df = screening[screening["split"] == split]
    targets = df.loc[df["hair_coverage"] <= CLEAN_MAX_COVERAGE, "image_id"]
    sources = df.loc[df["hair_coverage"] >= HAIR_MIN_COVERAGE, "image_id"]
    return sorted(targets.tolist()), sorted(sources.tolist())


class HairCompositeDataset(Dataset):
    """(composite, clean) pairs; items match `PairedDataset`: HR = clean, SR = composite."""

    def __init__(self, split: str, targets: list[str], sources: list[str],
                 images_dir: Path, layers_dir: Path, size: int = 128,
                 length: int | None = None, fixed: bool = False, seed: int = 0,
                 simple_prob: float = 0.3, max_shift: int = 120, max_sources: int = 3,
                 groups: dict[str, str] | None = None):
        if split not in ALLOWED_SPLITS:
            raise ValueError(f"split {split!r} is not allowed; only {ALLOWED_SPLITS}")
        self.split = split
        self.images_dir = Path(images_dir)
        self.layers_dir = Path(layers_dir)
        self.targets = list(targets)
        self.sources = [s for s in sources if (self.layers_dir / f"{s}.png").exists()]
        if not self.targets or not self.sources:
            raise ValueError("no targets or no harvested sources for this split")
        self.size, self.fixed, self.seed = int(size), bool(fixed), int(seed)
        self.simple_prob, self.max_shift = float(simple_prob), int(max_shift)
        self.max_sources = max(1, int(max_sources))
        self.length = int(length) if length else len(self.targets)
        self.groups = groups or {}
        self._rng_cache: np.random.Generator | None = None

    def __len__(self) -> int:
        return self.length

    def _rng(self, idx: int) -> np.random.Generator:
        if self.fixed:
            return np.random.default_rng([self.seed, int(idx)])
        if self._rng_cache is None:
            self._rng_cache = np.random.default_rng(
                [self.seed, os.getpid(), int(torch.initial_seed() % (2**31))])
        return self._rng_cache

    def compose(self, idx: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict]:
        """Full-resolution target, composite, soft alpha, and what was used."""
        rng = self._rng(idx)
        tid = self.targets[int(idx) % len(self.targets)]
        k = int(rng.integers(1, self.max_sources + 1))
        sids = [self.sources[int(rng.integers(len(self.sources)))] for _ in range(k)]
        target = np.asarray(Image.open(self.images_dir / f"{tid}.jpg").convert("RGB"))
        h, w = target.shape[:2]
        comp = target.astype(np.float32)
        alpha_tot = np.zeros((h, w), np.float32)
        for sid in sids:
            src = np.asarray(Image.open(self.images_dir / f"{sid}.jpg").convert("RGB"))
            mask = cv2.imread(str(self.layers_dir / f"{sid}.png"), cv2.IMREAD_GRAYSCALE)
            if src.shape[:2] != (h, w):
                src = cv2.resize(src, (w, h), interpolation=cv2.INTER_LINEAR)
            if mask.shape != (h, w):
                mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            if rng.random() < 0.5:
                src, mask = src[:, ::-1], mask[:, ::-1]
            if rng.random() < 0.5:
                src, mask = src[::-1], mask[::-1]
            dy = int(rng.integers(-self.max_shift, self.max_shift + 1))
            dx = int(rng.integers(-self.max_shift, self.max_shift + 1))
            src = np.roll(src, (dy, dx), axis=(0, 1))
            mask = np.roll(mask, (dy, dx), axis=(0, 1))
            m = (mask > 0).astype(np.uint8)
            m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
            alpha = np.clip(cv2.GaussianBlur(m.astype(np.float32), (0, 0), 0.7), 0.0, 1.0)
            a3 = alpha[..., None]
            comp = comp * (1 - a3) + src.astype(np.float32) * a3
            alpha_tot = 1.0 - (1.0 - alpha_tot) * (1.0 - alpha)
        comp = np.clip(np.rint(comp), 0, 255).astype(np.uint8)
        simple = False
        if self.simple_prob > 0 and rng.random() < self.simple_prob:
            hairy, _, _ = render_hair(Image.fromarray(comp), tid, int(rng.integers(2**31)))
            comp, simple = np.asarray(hairy), True
        return target, comp, alpha_tot, {"target": tid, "sources": sids, "simple": simple}

    def _resize(self, arr: np.ndarray) -> torch.Tensor:
        im = Image.fromarray(arr).resize((self.size, self.size), Image.Resampling.BICUBIC)
        x = np.asarray(im, dtype=np.float32)
        return torch.from_numpy(x / 127.5 - 1.0).permute(2, 0, 1)

    def __getitem__(self, idx: int):
        target, comp, _, _ = self.compose(idx)
        return {"HR": self._resize(target), "SR": self._resize(comp)}

    def manifest(self) -> dict:
        ids = sorted(set(self.targets) | set(self.sources))
        grp = {self.groups[i] for i in ids if i in self.groups}
        return {"n": int(self.length), "target_ids": sorted(self.targets),
                "source_ids": sorted(self.sources), "splits": [self.split],
                "groups": sorted(grp)}


def build_datasets(screening_csv: Path, images_dir: Path, layers_dir: Path,
                   size: int = 128, val_length: int = 192, seed: int = 0,
                   min_layer_fraction: float = MIN_LAYER_FRACTION):
    """Train and validation composite datasets from the screening table."""
    scr = pd.read_csv(screening_csv)
    layer_stats = pd.read_csv(Path(layers_dir) / "summary.csv")
    usable = set(layer_stats.loc[layer_stats["fraction"] >= min_layer_fraction, "image_id"])
    groups = dict(zip(scr["image_id"], scr["group_id"]))
    tr_t, tr_s = select_pools(scr, "train")
    va_t, va_s = select_pools(scr, "val")
    tr_s, va_s = [i for i in tr_s if i in usable], [i for i in va_s if i in usable]
    train = HairCompositeDataset("train", tr_t, tr_s, images_dir, layers_dir, size,
                                 fixed=False, seed=seed, groups=groups)
    val = HairCompositeDataset("val", va_t, va_s, images_dir, layers_dir, size,
                               length=min(val_length, len(va_t)), fixed=True,
                               seed=seed + 1, groups=groups)
    return train, val


## 5a. DM-AHR U-Net (vendored upstream)

The pinned upstream denoiser, unmodified.


In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F
from inspect import isfunction


def exists(x):
    return x is not None


def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d

# PositionalEncoding Source： https://github.com/lmnt-com/wavegrad/blob/master/src/wavegrad/model.py
class PositionalEncoding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, noise_level):
        count = self.dim // 2
        step = torch.arange(count, dtype=noise_level.dtype,
                            device=noise_level.device) / count
        encoding = noise_level.unsqueeze(
            1) * torch.exp(-math.log(1e4) * step.unsqueeze(0))
        encoding = torch.cat(
            [torch.sin(encoding), torch.cos(encoding)], dim=-1)
        return encoding


class FeatureWiseAffine(nn.Module):
    def __init__(self, in_channels, out_channels, use_affine_level=False):
        super(FeatureWiseAffine, self).__init__()
        self.use_affine_level = use_affine_level
        self.noise_func = nn.Sequential(
            nn.Linear(in_channels, out_channels*(1+self.use_affine_level))
        )

    def forward(self, x, noise_embed):
        batch = x.shape[0]
        if self.use_affine_level:
            gamma, beta = self.noise_func(noise_embed).view(
                batch, -1, 1, 1).chunk(2, dim=1)
            x = (1 + gamma) * x + beta
        else:
            x = x + self.noise_func(noise_embed).view(batch, -1, 1, 1)
        return x


class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)


class Upsample(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = nn.Conv2d(dim, dim, 3, padding=1)

    def forward(self, x):
        return self.conv(self.up(x))


class Downsample(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv2d(dim, dim, 3, 2, 1)

    def forward(self, x):
        return self.conv(x)


# building block modules


class Block(nn.Module):
    def __init__(self, dim, dim_out, groups=32, dropout=0):
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(groups, dim),
            Swish(),
            nn.Dropout(dropout) if dropout != 0 else nn.Identity(),
            nn.Conv2d(dim, dim_out, 3, padding=1)
        )

    def forward(self, x):
        return self.block(x)


class ResnetBlock(nn.Module):
    def __init__(self, dim, dim_out, noise_level_emb_dim=None, dropout=0, use_affine_level=False, norm_groups=32):
        super().__init__()
        self.noise_func = FeatureWiseAffine(
            noise_level_emb_dim, dim_out, use_affine_level)

        self.block1 = Block(dim, dim_out, groups=norm_groups)
        self.block2 = Block(dim_out, dim_out, groups=norm_groups, dropout=dropout)
        self.res_conv = nn.Conv2d(
            dim, dim_out, 1) if dim != dim_out else nn.Identity()

    def forward(self, x, time_emb):
        b, c, h, w = x.shape
        h = self.block1(x)
        h = self.noise_func(h, time_emb)
        h = self.block2(h)
        return h + self.res_conv(x)


class SelfAttention(nn.Module):
    def __init__(self, in_channel, n_head=1, norm_groups=32):
        super().__init__()

        self.n_head = n_head

        self.norm = nn.GroupNorm(norm_groups, in_channel)
        self.qkv = nn.Conv2d(in_channel, in_channel * 3, 1, bias=False)
        self.out = nn.Conv2d(in_channel, in_channel, 1)

    def forward(self, input):
        batch, channel, height, width = input.shape
        n_head = self.n_head
        head_dim = channel // n_head

        norm = self.norm(input)
        qkv = self.qkv(norm).view(batch, n_head, head_dim * 3, height, width)
        query, key, value = qkv.chunk(3, dim=2)  # bhdyx

        attn = torch.einsum(
            "bnchw, bncyx -> bnhwyx", query, key
        ).contiguous() / math.sqrt(channel)
        attn = attn.view(batch, n_head, height, width, -1)
        attn = torch.softmax(attn, -1)
        attn = attn.view(batch, n_head, height, width, height, width)

        out = torch.einsum("bnhwyx, bncyx -> bnchw", attn, value).contiguous()
        out = self.out(out.view(batch, channel, height, width))

        return out + input


class ResnetBlocWithAttn(nn.Module):
    def __init__(self, dim, dim_out, *, noise_level_emb_dim=None, norm_groups=32, dropout=0, with_attn=False):
        super().__init__()
        self.with_attn = with_attn
        self.res_block = ResnetBlock(
            dim, dim_out, noise_level_emb_dim, norm_groups=norm_groups, dropout=dropout)
        if with_attn:
            self.attn = SelfAttention(dim_out, norm_groups=norm_groups)

    def forward(self, x, time_emb):
        x = self.res_block(x, time_emb)
        if(self.with_attn):
            x = self.attn(x)
        return x


class UNet(nn.Module):
    def __init__(
        self,
        in_channel=6,
        out_channel=3,
        inner_channel=32,
        norm_groups=32,
        channel_mults=(1, 2, 4, 8, 8),
        attn_res=(8),
        res_blocks=3,
        dropout=0,
        with_noise_level_emb=True,
        image_size=128
    ):
        super().__init__()

        if with_noise_level_emb:
            noise_level_channel = inner_channel
            self.noise_level_mlp = nn.Sequential(
                PositionalEncoding(inner_channel),
                nn.Linear(inner_channel, inner_channel * 4),
                Swish(),
                nn.Linear(inner_channel * 4, inner_channel)
            )
        else:
            noise_level_channel = None
            self.noise_level_mlp = None

        num_mults = len(channel_mults)
        pre_channel = inner_channel
        feat_channels = [pre_channel]
        now_res = image_size
        downs = [nn.Conv2d(in_channel, inner_channel,
                           kernel_size=3, padding=1)]
        for ind in range(num_mults):
            is_last = (ind == num_mults - 1)
            use_attn = (now_res in attn_res)
            channel_mult = inner_channel * channel_mults[ind]
            for _ in range(0, res_blocks):
                downs.append(ResnetBlocWithAttn(
                    pre_channel, channel_mult, noise_level_emb_dim=noise_level_channel, norm_groups=norm_groups, dropout=dropout, with_attn=use_attn))
                feat_channels.append(channel_mult)
                pre_channel = channel_mult
            if not is_last:
                downs.append(Downsample(pre_channel))
                feat_channels.append(pre_channel)
                now_res = now_res//2
        self.downs = nn.ModuleList(downs)

        self.mid = nn.ModuleList([
            ResnetBlocWithAttn(pre_channel, pre_channel, noise_level_emb_dim=noise_level_channel, norm_groups=norm_groups,
                               dropout=dropout, with_attn=True),
            ResnetBlocWithAttn(pre_channel, pre_channel, noise_level_emb_dim=noise_level_channel, norm_groups=norm_groups,
                               dropout=dropout, with_attn=False)
        ])

        ups = []
        for ind in reversed(range(num_mults)):
            is_last = (ind < 1)
            use_attn = (now_res in attn_res)
            channel_mult = inner_channel * channel_mults[ind]
            for _ in range(0, res_blocks+1):
                ups.append(ResnetBlocWithAttn(
                    pre_channel+feat_channels.pop(), channel_mult, noise_level_emb_dim=noise_level_channel, norm_groups=norm_groups,
                        dropout=dropout, with_attn=use_attn))
                pre_channel = channel_mult
            if not is_last:
                ups.append(Upsample(pre_channel))
                now_res = now_res*2

        self.ups = nn.ModuleList(ups)

        self.final_conv = Block(pre_channel, default(out_channel, in_channel), groups=norm_groups)

    def forward(self, x, time):
        t = self.noise_level_mlp(time) if exists(
            self.noise_level_mlp) else None

        feats = []
        for layer in self.downs:
            if isinstance(layer, ResnetBlocWithAttn):
                x = layer(x, t)
            else:
                x = layer(x)
            feats.append(x)

        for layer in self.mid:
            if isinstance(layer, ResnetBlocWithAttn):
                x = layer(x, t)
            else:
                x = layer(x)

        for layer in self.ups:
            if isinstance(layer, ResnetBlocWithAttn):
                x = layer(torch.cat((x, feats.pop()), dim=1), t)
            else:
                x = layer(x)

        return self.final_conv(x)


## 5b. Diffusion schedule and losses (vendored upstream)

The pinned upstream GaussianDiffusion. Note the noise-level representation it expects.


In [ ]:
import math
import torch
from torch import device, nn, einsum
import torch.nn.functional as F
from inspect import isfunction
from functools import partial
import numpy as np
from tqdm import tqdm


def _warmup_beta(linear_start, linear_end, n_timestep, warmup_frac):
    betas = linear_end * np.ones(n_timestep, dtype=np.float64)
    warmup_time = int(n_timestep * warmup_frac)
    betas[:warmup_time] = np.linspace(
        linear_start, linear_end, warmup_time, dtype=np.float64)
    return betas


def make_beta_schedule(schedule, n_timestep, linear_start=1e-4, linear_end=2e-2, cosine_s=8e-3):
    if schedule == 'quad':
        betas = np.linspace(linear_start ** 0.5, linear_end ** 0.5,
                            n_timestep, dtype=np.float64) ** 2
    elif schedule == 'linear':
        betas = np.linspace(linear_start, linear_end,
                            n_timestep, dtype=np.float64)
    elif schedule == 'warmup10':
        betas = _warmup_beta(linear_start, linear_end,
                             n_timestep, 0.1)
    elif schedule == 'warmup50':
        betas = _warmup_beta(linear_start, linear_end,
                             n_timestep, 0.5)
    elif schedule == 'const':
        betas = linear_end * np.ones(n_timestep, dtype=np.float64)
    elif schedule == 'jsd':  # 1/T, 1/(T-1), 1/(T-2), ..., 1
        betas = 1. / np.linspace(n_timestep,
                                 1, n_timestep, dtype=np.float64)
    elif schedule == "cosine":
        timesteps = (
            torch.arange(n_timestep + 1, dtype=torch.float64) /
            n_timestep + cosine_s
        )
        alphas = timesteps / (1 + cosine_s) * math.pi / 2
        alphas = torch.cos(alphas).pow(2)
        alphas = alphas / alphas[0]
        betas = 1 - alphas[1:] / alphas[:-1]
        betas = betas.clamp(max=0.999)
    else:
        raise NotImplementedError(schedule)
    return betas


# gaussian diffusion trainer class

def exists(x):
    return x is not None


def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d


class GaussianDiffusion(nn.Module):
    def __init__(
        self,
        denoise_fn,
        image_size,
        channels=3,
        loss_type='l1',
        conditional=True,
        schedule_opt=None
    ):
        super().__init__()
        self.channels = channels
        self.image_size = image_size
        self.denoise_fn = denoise_fn
        self.loss_type = loss_type
        self.conditional = conditional
        self._upstream_skipped_schedule = schedule_opt  # upstream skips buffer registration
            # self.set_new_noise_schedule(schedule_opt)

    def set_loss(self, device):
        if self.loss_type == 'l1':
            self.loss_func = nn.L1Loss(reduction='sum').to(device)
        elif self.loss_type == 'l2':
            self.loss_func = nn.MSELoss(reduction='sum').to(device)
        else:
            raise NotImplementedError()

    def set_new_noise_schedule(self, schedule_opt, device):
        to_torch = partial(torch.tensor, dtype=torch.float32, device=device)

        betas = make_beta_schedule(
            schedule=schedule_opt['schedule'],
            n_timestep=schedule_opt['n_timestep'],
            linear_start=schedule_opt['linear_start'],
            linear_end=schedule_opt['linear_end'])
        betas = betas.detach().cpu().numpy() if isinstance(
            betas, torch.Tensor) else betas
        alphas = 1. - betas
        alphas_cumprod = np.cumprod(alphas, axis=0)
        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])
        self.sqrt_alphas_cumprod_prev = np.sqrt(
            np.append(1., alphas_cumprod))

        timesteps, = betas.shape
        self.num_timesteps = int(timesteps)
        self.register_buffer('betas', to_torch(betas))
        self.register_buffer('alphas_cumprod', to_torch(alphas_cumprod))
        self.register_buffer('alphas_cumprod_prev',
                             to_torch(alphas_cumprod_prev))

        # calculations for diffusion q(x_t | x_{t-1}) and others
        self.register_buffer('sqrt_alphas_cumprod',
                             to_torch(np.sqrt(alphas_cumprod)))
        self.register_buffer('sqrt_one_minus_alphas_cumprod',
                             to_torch(np.sqrt(1. - alphas_cumprod)))
        self.register_buffer('log_one_minus_alphas_cumprod',
                             to_torch(np.log(1. - alphas_cumprod)))
        self.register_buffer('sqrt_recip_alphas_cumprod',
                             to_torch(np.sqrt(1. / alphas_cumprod)))
        self.register_buffer('sqrt_recipm1_alphas_cumprod',
                             to_torch(np.sqrt(1. / alphas_cumprod - 1)))

        # calculations for posterior q(x_{t-1} | x_t, x_0)
        posterior_variance = betas * \
            (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        # above: equal to 1. / (1. / (1. - alpha_cumprod_tm1) + alpha_t / beta_t)
        self.register_buffer('posterior_variance',
                             to_torch(posterior_variance))
        # below: log calculation clipped because the posterior variance is 0 at the beginning of the diffusion chain
        self.register_buffer('posterior_log_variance_clipped', to_torch(
            np.log(np.maximum(posterior_variance, 1e-20))))
        self.register_buffer('posterior_mean_coef1', to_torch(
            betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))
        self.register_buffer('posterior_mean_coef2', to_torch(
            (1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))

    def predict_start_from_noise(self, x_t, t, noise):
        return self.sqrt_recip_alphas_cumprod[t] * x_t - \
            self.sqrt_recipm1_alphas_cumprod[t] * noise

    def q_posterior(self, x_start, x_t, t):
        posterior_mean = self.posterior_mean_coef1[t] * \
            x_start + self.posterior_mean_coef2[t] * x_t
        posterior_log_variance_clipped = self.posterior_log_variance_clipped[t]
        return posterior_mean, posterior_log_variance_clipped

    def p_mean_variance(self, x, t, clip_denoised: bool, condition_x=None):
        batch_size = x.shape[0]
        noise_level = torch.FloatTensor(
            [self.sqrt_alphas_cumprod_prev[t+1]]).repeat(batch_size, 1).to(x.device)
        if condition_x is not None:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.denoise_fn(torch.cat([condition_x, x], dim=1), noise_level))
        else:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.denoise_fn(x, noise_level))

        if clip_denoised:
            x_recon.clamp_(-1., 1.)

        model_mean, posterior_log_variance = self.q_posterior(
            x_start=x_recon, x_t=x, t=t)
        return model_mean, posterior_log_variance

    @torch.no_grad()
    def p_sample(self, x, t, clip_denoised=True, condition_x=None):
        model_mean, model_log_variance = self.p_mean_variance(
            x=x, t=t, clip_denoised=clip_denoised, condition_x=condition_x)
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        return model_mean + noise * (0.5 * model_log_variance).exp()

    @torch.no_grad()
    def p_sample_loop(self, x_in, continous=False):
        device = self.betas.device
        sample_inter = (1 | (self.num_timesteps//10))
        if not self.conditional:
            shape = x_in
            img = torch.randn(shape, device=device)
            ret_img = img
            for i in tqdm(reversed(range(0, self.num_timesteps)), desc='sampling loop time step', total=self.num_timesteps):
                img = self.p_sample(img, i)
                if i % sample_inter == 0:
                    ret_img = torch.cat([ret_img, img], dim=0)
        else:
            x = x_in
            shape = x.shape
            img = torch.randn(shape, device=device)
            ret_img = x
            for i in tqdm(reversed(range(0, self.num_timesteps)), desc='sampling loop time step', total=self.num_timesteps):
                img = self.p_sample(img, i, condition_x=x)
                if i % sample_inter == 0:
                    ret_img = torch.cat([ret_img, img], dim=0)
        if continous:
            return ret_img
        else:
            return ret_img[-1]

    @torch.no_grad()
    def sample(self, batch_size=1, continous=False):
        image_size = self.image_size
        channels = self.channels
        return self.p_sample_loop((batch_size, channels, image_size, image_size), continous)

    @torch.no_grad()
    def super_resolution(self, x_in, continous=False):
        return self.p_sample_loop(x_in, continous)

    def q_sample(self, x_start, continuous_sqrt_alpha_cumprod, noise=None):
        noise = default(noise, lambda: torch.randn_like(x_start))

        # random gama
        return (
            continuous_sqrt_alpha_cumprod * x_start +
            (1 - continuous_sqrt_alpha_cumprod**2).sqrt() * noise
        )

    def p_losses(self, x_in, noise=None):
        x_start = x_in['HR']
        [b, c, h, w] = x_start.shape
        t = np.random.randint(1, self.num_timesteps + 1)
        continuous_sqrt_alpha_cumprod = torch.FloatTensor(
            np.random.uniform(
                self.sqrt_alphas_cumprod_prev[t-1],
                self.sqrt_alphas_cumprod_prev[t],
                size=b
            )
        ).to(x_start.device)
        continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(
            b, -1)

        noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(
            x_start=x_start, continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1, 1), noise=noise)

        if not self.conditional:
            x_recon = self.denoise_fn(x_noisy, continuous_sqrt_alpha_cumprod)
        else:
            x_recon = self.denoise_fn(
                torch.cat([x_in['SR'], x_noisy], dim=1), continuous_sqrt_alpha_cumprod)

        loss = self.loss_func(noise, x_recon)
        return loss

    def forward(self, x, *args, **kwargs):
        return self.p_losses(x, *args, **kwargs)


## 5c. Model factory (vendored upstream)

Builds the U-Net and wraps it in GaussianDiffusion.


In [ ]:
import functools
import logging
import torch
import torch.nn as nn
from torch.nn import init
from torch.nn import modules
logger = logging.getLogger('base')
####################
# initialize
####################


def weights_init_normal(m, std=0.02):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        init.normal_(m.weight.data, 0.0, std)
        if m.bias is not None:
            m.bias.data.zero_()
    elif classname.find('Linear') != -1:
        init.normal_(m.weight.data, 0.0, std)
        if m.bias is not None:
            m.bias.data.zero_()
    elif classname.find('BatchNorm2d') != -1:
        init.normal_(m.weight.data, 1.0, std)  # BN also uses norm
        init.constant_(m.bias.data, 0.0)


def weights_init_kaiming(m, scale=1):
    classname = m.__class__.__name__
    if classname.find('Conv2d') != -1:
        init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
        m.weight.data *= scale
        if m.bias is not None:
            m.bias.data.zero_()
    elif classname.find('Linear') != -1:
        init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
        m.weight.data *= scale
        if m.bias is not None:
            m.bias.data.zero_()
    elif classname.find('BatchNorm2d') != -1:
        init.constant_(m.weight.data, 1.0)
        init.constant_(m.bias.data, 0.0)


def weights_init_orthogonal(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        init.orthogonal_(m.weight.data, gain=1)
        if m.bias is not None:
            m.bias.data.zero_()
    elif classname.find('Linear') != -1:
        init.orthogonal_(m.weight.data, gain=1)
        if m.bias is not None:
            m.bias.data.zero_()
    elif classname.find('BatchNorm2d') != -1:
        init.constant_(m.weight.data, 1.0)
        init.constant_(m.bias.data, 0.0)


def init_weights(net, init_type='kaiming', scale=1, std=0.02):
    # scale for 'kaiming', std for 'normal'.
    logger.info('Initialization method [{:s}]'.format(init_type))
    if init_type == 'normal':
        weights_init_normal_ = functools.partial(weights_init_normal, std=std)
        net.apply(weights_init_normal_)
    elif init_type == 'kaiming':
        weights_init_kaiming_ = functools.partial(
            weights_init_kaiming, scale=scale)
        net.apply(weights_init_kaiming_)
    elif init_type == 'orthogonal':
        net.apply(weights_init_orthogonal)
    else:
        raise NotImplementedError(
            'initialization method [{:s}] not implemented'.format(init_type))


####################
# define network
####################


# Generator
def define_G(opt):
    model_opt = opt['model']
    if ('norm_groups' not in model_opt['unet']) or model_opt['unet']['norm_groups'] is None:
        model_opt['unet']['norm_groups']=32
    model = UNet(
        in_channel=model_opt['unet']['in_channel'],
        out_channel=model_opt['unet']['out_channel'],
        norm_groups=model_opt['unet']['norm_groups'],
        inner_channel=model_opt['unet']['inner_channel'],
        channel_mults=model_opt['unet']['channel_multiplier'],
        attn_res=model_opt['unet']['attn_res'],
        res_blocks=model_opt['unet']['res_blocks'],
        dropout=model_opt['unet']['dropout'],
        image_size=model_opt['diffusion']['image_size']
    )
    netG = GaussianDiffusion(
        model,
        image_size=model_opt['diffusion']['image_size'],
        channels=model_opt['diffusion']['channels'],
        loss_type='l1',    # L1 or L2
        conditional=model_opt['diffusion']['conditional'],
        schedule_opt=model_opt['beta_schedule']['train']
    )
    if opt['phase'] == 'train':
        # init_weights(netG, init_type='kaiming', scale=0.1)
        init_weights(netG, init_type='orthogonal')
    if opt['gpu_ids'] and opt['distributed']:
        assert torch.cuda.is_available()
        netG = nn.DataParallel(netG)
    return netG


## 5d. Adapter around the upstream model

Loads checkpoints strictly and implements the production respaced DDIM sampler.


In [ ]:
"""Adapter around the pinned upstream DM-AHR/SR3 implementation.

The upstream tree is vendored in `third_party/dmahr` at a pinned commit so the
audit and the executable code cannot drift apart.
"""


import json
import re
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image


UPSTREAM_DIR = REPO_ROOT / "third_party" / "dmahr"
CHECKPOINT_NAME = "I725000_E1083_gen.pth"


def _ensure_path() -> None:
    p = str(UPSTREAM_DIR)
    if p not in sys.path:
        sys.path.insert(0, p)


def _strip_comments(text: str) -> str:
    """Make the upstream config JSON-parseable.

    `config/*.json` in the pinned repo is JSON5: it has `//` and `/* */`
    comments plus trailing commas, so plain `json.load` rejects it.
    """
    out: list[str] = []
    i, n = 0, len(text)
    in_string = False
    escaped = False
    while i < n:
        ch = text[i]
        if in_string:
            out.append(ch)
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            i += 1
            continue
        if ch == '"':
            in_string = True
            out.append(ch)
            i += 1
            continue
        if ch == "/" and i + 1 < n and text[i + 1] == "/":
            j = text.find("\n", i)
            i = n if j < 0 else j
            continue
        if ch == "/" and i + 1 < n and text[i + 1] == "*":
            j = text.find("*/", i + 2)
            i = n if j < 0 else j + 2
            continue
        out.append(ch)
        i += 1
    return re.sub(r",(\s*[}\]])", r"\1", "".join(out))


def load_upstream_config(name: str = "akiec.json") -> dict:
    return json.loads(json.dumps(CONFIGS[name]))


def build_opt(cfg: dict, upstream: dict | None = None) -> dict:
    upstream = upstream or load_upstream_config()
    keep = upstream.get("model", {})
    opt = {
        "name": "dmahr",
        "phase": "val",
        "gpu_ids": [],
        "path": {"log": "/tmp", "tb_logger": "/tmp", "results": "/tmp",
                 "checkpoint": "/tmp", "resume_state": None},
        "datasets": {},
        "model": {
            "which_model_G": keep.get("which_model_G", "sr3"),
            "finetune_norm": False,
            "unet": dict(cfg["unet"]),
            "beta_schedule": {
                "train": {**cfg["beta_schedule"], "n_timestep": int(cfg["num_timesteps"])},
                "val": {**cfg["beta_schedule"], "n_timestep": int(cfg["num_timesteps"])},
            },
            "diffusion": {
                "image_size": int(cfg["native_resolution"]),
                "channels": 3,
                "conditional": True,
            },
        },
        "train": {
            "n_iter": int(cfg["train"]["iterations"]),
            "val_freq": 1e9,
            "save_checkpoint_freq": 1e9,
            "print_freq": 100,
            "optimizer": {"type": cfg["train"]["optimizer"], "lr": float(cfg["train"]["lr"])},
            "ema_scheduler": {},
        },
        "wandb": {},
        "distributed": False,
    }
    return opt


def opt_hash(opt: dict) -> str:
    return config_hash(opt)


def load_model(cfg: dict, checkpoint: Path, device: str = "cuda") -> torch.nn.Module:
    _ensure_path()

    opt = build_opt(cfg)
    net = define_G(opt)
    # The upstream __init__ deliberately skips buffer registration (the call is
    # commented out), so the diffusion buffers only exist after an explicit
    # schedule call. Register them first, then load so strict=True can match.
    net.set_new_noise_schedule(
        {"schedule": cfg["beta_schedule"]["schedule"],
         "linear_start": cfg["beta_schedule"]["linear_start"],
         "linear_end": cfg["beta_schedule"]["linear_end"],
         "n_timestep": int(cfg["num_timesteps"])},
        device=torch.device("cpu"),
    )
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if any(k.startswith("module.") for k in state):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
    net.load_state_dict(state, strict=True)
    net.eval()
    net.to(device)
    net.num_timesteps = int(cfg["num_timesteps"])
    net.set_new_noise_schedule(
        {"schedule": cfg["beta_schedule"]["schedule"],
         "linear_start": cfg["beta_schedule"]["linear_start"],
         "linear_end": cfg["beta_schedule"]["linear_end"],
         "n_timestep": int(cfg["num_timesteps"])},
        device=torch.device(device),
    )
    print(f"loaded DM-AHR checkpoint {checkpoint} (strict)", flush=True)
    return net


def preprocess(images: list[Image.Image], size: int = 128) -> torch.Tensor:
    arrs = []
    for im in images:
        a = np.asarray(im.convert("RGB").resize((size, size), Image.Resampling.BICUBIC),
                       dtype=np.float32)
        arrs.append((a / 127.5) - 1.0)
    return torch.from_numpy(np.stack(arrs)).permute(0, 3, 1, 2).contiguous()


def postprocess(tensor: torch.Tensor) -> list[Image.Image]:
    arr = tensor.detach().float().cpu().numpy()
    if arr.ndim == 3:
        arr = arr[None]
    arr = np.clip((arr + 1.0) * 127.5, 0, 255).round().astype(np.uint8)
    return [Image.fromarray(a.transpose(1, 2, 0)) for a in arr]


@torch.no_grad()
def clean_batch(
    net: torch.nn.Module,
    images: list[Image.Image],
    cfg: dict,
    device: str = "cuda",
    seed: int | None = None,
    num_timesteps: int | None = None,
    use_bf16: bool = False,
    respaced: bool = False,
    schedule: str | None = None,
    start_t: int | None = None,
) -> tuple[list[Image.Image], dict]:
    """Run DM-AHR on a batch of images. Returns cleaned images and run stats."""
    size = int(cfg["native_resolution"])
    x = preprocess(images, size).to(device)
    if respaced:
        # Few-step DDIM over the trained 2,000-step schedule. Truncating the
        # schedule instead (num_timesteps=25) barely adds noise and produces
        # garbage; striding the trained schedule is what makes 25 steps work.
        out, stats = respaced_super_resolution(
            net, x, steps=int(num_timesteps or 25), seed=seed, use_bf16=use_bf16,
            start_t=start_t,
        )
        stats.update({"respaced": True, "num_timesteps": int(net.num_timesteps),
                      "steps": int(num_timesteps or 25), "start_t": start_t,
                      "bf16": bool(use_bf16)})
        return postprocess(out), stats
    if num_timesteps is not None and int(num_timesteps) != int(net.num_timesteps):
        net.num_timesteps = int(num_timesteps)
        net.set_new_noise_schedule(
            {"schedule": schedule or cfg["beta_schedule"]["schedule"],
             "linear_start": cfg["beta_schedule"]["linear_start"],
             "linear_end": cfg["beta_schedule"]["linear_end"],
             "n_timestep": int(num_timesteps)},
            device=torch.device(device),
        )
    if seed is not None:
        torch.manual_seed(int(seed))
        torch.cuda.manual_seed_all(int(seed))
    # Upstream's `continous=False` path returns `ret_img[-1]`, which silently
    # collapses a batch to its last image. Ask for the full trajectory and take
    # the final B rows, which is the last denoised set for the whole batch.
    # bf16 autocast roughly halves the cost of the 2,000 sequential UNet calls
    # on Ampere tensor cores; the sampler is compute bound, not memory bound.
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
        trajectory = net.super_resolution(x, continous=True)
    out = trajectory[-x.shape[0]:]
    finite = bool(torch.isfinite(out).all())
    stats = {
        "num_timesteps": int(net.num_timesteps),
        "finite": finite,
        "shape": list(out.shape),
        "min": float(out.min()),
        "max": float(out.max()),
        "seed": seed,
        "bf16": use_bf16,
        "batch_returned": int(out.shape[0]),
        "batch_requested": int(x.shape[0]),
    }
    if not finite:
        raise FloatingPointError("DM-AHR produced non-finite values")
    if out.shape[0] != x.shape[0]:
        raise RuntimeError("DM-AHR returned the wrong number of images")
    return postprocess(out), stats


@torch.no_grad()
def respaced_super_resolution(net, cond: torch.Tensor, steps: int = 25,
                              seed: int | None = None,
                              use_bf16: bool = False,
                              start_t: int | None = None) -> tuple[torch.Tensor, dict]:
    """Deterministic DDIM sampling over the trained schedule in `steps` jumps.

    Uses the model's own 2,000-step alphas and visits `steps` of them evenly
    spaced, so every denoiser call is made at a noise level the model was
    trained on. This is what makes 25-step sampling usable.

    `start_t` starts the chain at that schedule index from the noised
    conditioning image instead of from pure noise. Hair is thin, so a chain
    that begins at a middling noise level keeps the global tone of the input and
    never visits the high-noise levels where a briefly trained model is weak.
    """
    if seed is not None:
        torch.manual_seed(int(seed))
        torch.cuda.manual_seed_all(int(seed))
    T = int(net.num_timesteps)
    top = T - 1 if start_t is None else int(min(max(start_t, 1), T - 1))
    seq = np.linspace(top, 0, steps).round().astype(int)
    seq = np.unique(seq)[::-1]  # descending, no duplicates
    noise = torch.randn(cond.shape, device=cond.device)
    if start_t is None:
        x = noise
    else:
        a = net.alphas_cumprod[top]
        x = a.sqrt() * cond + (1.0 - a).sqrt() * noise
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
        for i, t in enumerate(seq):
            level = torch.FloatTensor(
                [net.sqrt_alphas_cumprod_prev[t + 1]]).repeat(cond.shape[0], 1).to(cond.device)
            eps = net.denoise_fn(torch.cat([cond, x], dim=1), level)
            x0 = (net.sqrt_recip_alphas_cumprod[t] * x
                  - net.sqrt_recipm1_alphas_cumprod[t] * eps).clamp(-1.0, 1.0)
            if i == len(seq) - 1:
                x = x0
                break
            t_next = int(seq[i + 1])
            ac_next = net.alphas_cumprod[t_next]
            # Deterministic DDIM step: re-noise x0 to the next scheduled level.
            x = ac_next.sqrt() * x0 + (1.0 - ac_next).sqrt() * eps
    stats = {"finite": bool(torch.isfinite(x).all()),
             "shape": list(x.shape), "timesteps_visited": seq.tolist()}
    if not stats["finite"]:
        raise FloatingPointError("DM-AHR produced non-finite values")
    return x, stats


## 5e. DM-AHR training loop

Repaired trainer with sampler-compatible noise-level encoding; resumes from train_state.pt.


In [ ]:
"""Leakage-controlled DM-AHR training on train-split paired sources only."""


import json
import hashlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset


log = setup_logging()


def _resolve_manifest_path(path: str) -> Path:
    """Re-root a recorded modal-volume path onto the current DATA_ROOT.

    Pair manifests were written inside a container, so they hold absolute
    `/__modal/volumes/<id>/data/...` paths that are void on any other volume.
    Splitting on the `/data/` marker keeps them usable.
    """
    p = Path(path)
    if p.exists():
        return p
    marker = "/data/"
    if marker in str(p):
        return data_root() / str(p).split(marker, 1)[1]
    return p


def _manifest_hash(pairs: pd.DataFrame) -> str:
    """SHA-256 over the ordered pair ids, so a run records exactly what it saw."""
    ids = sorted(str(i) for i in pairs["pair_id"])
    return hashlib.sha256("\n".join(ids).encode()).hexdigest()


class PairedDataset(Dataset):
    """(hairy, hairless) pairs at native resolution, in [-1, 1]."""

    def __init__(self, pairs: pd.DataFrame, size: int = 128):
        self.pairs = pairs.reset_index(drop=True)
        self.size = size

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        r = self.pairs.iloc[idx]
        hr = np.asarray(
            Image.open(_resolve_manifest_path(r["reference_path"])).convert("RGB").resize(
                (self.size, self.size), Image.Resampling.BICUBIC
            ),
            dtype=np.float32,
        )
        cond = np.asarray(
            Image.open(_resolve_manifest_path(r["hairy_path"])).convert("RGB").resize(
                (self.size, self.size), Image.Resampling.BICUBIC
            ),
            dtype=np.float32,
        )
        if idx % 2 == 1:
            hr, cond = hr[:, ::-1].copy(), cond[:, ::-1].copy()
        return {
            "HR": torch.from_numpy(hr / 127.5 - 1.0).permute(2, 0, 1),
            "SR": torch.from_numpy(cond / 127.5 - 1.0).permute(2, 0, 1),
        }


def _infinite(loader):
    while True:
        for batch in loader:
            yield batch


@torch.no_grad()
def val_loss(net, loader, device: str, timesteps=(200, 800, 1400, 1900)) -> float:
    net.eval()
    losses = []
    for batch in loader:
        hr = batch["HR"].to(device)
        cond = batch["SR"].to(device)
        for t in timesteps:
            # Upstream encodes the noise level as sqrt(alpha_cumprod_prev), not as
            # t/T: `NoiseLevelEncoder` multiplies a frequency ramp by this value,
            # and `p_sample` feeds the same quantity at inference. Using t/T here
            # trains one encoding and samples with another.
            alpha = net.sqrt_alphas_cumprod_prev[int(t) - 1]
            level = torch.full((hr.shape[0], 1), float(alpha), device=device)
            noise = torch.randn_like(hr)
            noisy = alpha * hr + (1 - alpha**2) ** 0.5 * noise
            pred = net.denoise_fn(torch.cat([cond, noisy], dim=1), level)
            losses.append(float((pred - noise).abs().mean()))
    net.train()
    return float(np.mean(losses))


def training_loss(net, hr: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
    """One training loss, equivalent to upstream `p_losses` / numel.

    RNG draw order (numpy t, numpy levels, torch noise) matches upstream so a
    seeded differential test can compare the two exactly.
    """
    t = np.random.randint(1, net.num_timesteps + 1)
    # Continuous noise level per sample, as upstream: uniform between
    # sqrt(alpha_cumprod_prev) at t-1 and at t.
    lo = float(net.sqrt_alphas_cumprod_prev[t - 1])
    hi = float(net.sqrt_alphas_cumprod_prev[t])
    alpha = torch.tensor(
        np.random.uniform(lo, hi, size=hr.shape[0]), dtype=hr.dtype, device=hr.device
    )
    alpha_4d = alpha.view(-1, 1, 1, 1)
    noise = torch.randn_like(hr)
    x_noisy = alpha_4d * hr + (1 - alpha_4d**2) ** 0.5 * noise
    pred = net.denoise_fn(torch.cat([cond, x_noisy], dim=1), alpha.view(-1, 1))
    return (pred - noise).abs().mean()


def _describe(ds) -> dict:
    """What a dataset contains, for the training manifest (pair table or composite)."""
    if hasattr(ds, "manifest"):
        m = ds.manifest()
        return {"n": m["n"], "ids": m["target_ids"] + m["source_ids"],
                "splits": m["splits"], "groups": set(m["groups"]),
                "hash": hashlib.sha256("\n".join(sorted(m["target_ids"] + m["source_ids"]))
                                       .encode()).hexdigest()}
    p = ds.pairs
    return {"n": int(len(ds)), "ids": p["pair_id"].tolist(),
            "splits": sorted(p["split"].unique().tolist()),
            "groups": set(p["group_id"]), "hash": _manifest_hash(p)}


def train(
    cfg: dict,
    pairs: pd.DataFrame | None,
    val_pairs: pd.DataFrame | None,
    out_dir: Path,
    device: str = "cuda",
    overfit: bool = False,
    init_from: Path | None = None,
    max_iterations: int | None = None,
    train_ds: Dataset | None = None,
    val_ds: Dataset | None = None,
    num_workers: int = 4,
    grad_clip: float | None = 1.0,
) -> dict:
    _ensure_path()

    out_dir.mkdir(parents=True, exist_ok=True)
    opt = build_opt(cfg)
    net = define_G(opt).to(device)
    # Upstream's define_G/__init__ never registers the diffusion buffers, so the
    # noise schedule must be set explicitly before any q_sample/p_losses use.
    net.set_new_noise_schedule(
        {"schedule": cfg["beta_schedule"]["schedule"],
         "linear_start": cfg["beta_schedule"]["linear_start"],
         "linear_end": cfg["beta_schedule"]["linear_end"],
         "n_timestep": int(cfg["num_timesteps"])},
        device=torch.device(device),
    )
    opt_g = torch.optim.Adam(net.parameters(), lr=float(cfg["train"]["lr"]))
    iterations = int(cfg["train"]["iterations"])
    if overfit and pairs is not None:
        pairs = pairs.head(int(cfg["train"].get("overfit_pairs", 12)))
        iterations = 500
    if max_iterations is not None:
        iterations = int(max_iterations)
    if train_ds is None:
        train_ds = PairedDataset(pairs, int(cfg["native_resolution"]))
    if val_ds is None:
        val_ds = PairedDataset(val_pairs, int(cfg["native_resolution"]))
    bs = int(cfg["train"]["batch_size"])
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True,
                              num_workers=int(num_workers), pin_memory=True,
                              persistent_workers=int(num_workers) > 0)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=2)

    state_path = out_dir / ("overfit_state.pt" if overfit else "train_state.pt")
    start = 0
    best = float("inf")
    history = []
    if init_from is not None:
        # Warm start from an existing checkpoint. Used to check that the fixed
        # trainer does not degrade weights it did not produce.
        init_state = torch.load(init_from, map_location="cpu", weights_only=True)
        if any(k.startswith("module.") for k in init_state):
            init_state = {k.replace("module.", "", 1): v for k, v in init_state.items()}
        net.load_state_dict(init_state, strict=True)
        log.info("warm-started from %s", init_from)
    if state_path.exists():
        state = torch.load(state_path, map_location="cpu", weights_only=False)
        net.load_state_dict(state["model"])
        opt_g.load_state_dict(state["optimizer"])
        start = int(state["iteration"])
        best = float(state.get("best_val", best))
        history = state.get("history", [])
        log.info("resumed DM-AHR at iteration %d", start)

    d_tr, d_va = _describe(train_ds), _describe(val_ds)
    save_json(
        {
            "config": cfg,
            "opt_hash": opt_hash(opt),
            "train_pairs": d_tr["n"],
            "val_pairs": d_va["n"],
            "train_pair_ids": d_tr["ids"],
            "val_pair_ids": d_va["ids"],
            "train_sources_splits": d_tr["splits"],
            "val_sources_splits": d_va["splits"],
            "leakage_check": bool(not d_tr["groups"] & d_va["groups"]),
            "overfit": overfit,
            # Provenance recorded at run time so a later audit does not have to
            # infer which noise-level convention or which data this run used.
            "noise_level_convention": "sqrt_alphas_cumprod_prev per sample, "
                                      "continuous in [t-1, t] (upstream p_losses)",
            "num_timesteps": int(net.num_timesteps),
            "ema": "none",
            "grad_clip": float(grad_clip) if grad_clip else None,
            "batch_size": int(cfg["train"]["batch_size"]),
            "init_from": str(init_from) if init_from is not None else None,
            "train_manifest_sha256": d_tr["hash"],
            "dataset_kind": type(train_ds).__name__,
        },
        out_dir / "training_manifest.json",
    )

    net.train()
    it = _infinite(train_loader)
    ckpt_every = int(cfg["train"]["checkpoint_every"])
    val_every = int(cfg["train"]["val_every"])
    t0 = time.time()
    for step in range(start + 1, iterations + 1):
        batch = next(it)
        hr = batch["HR"].to(device, non_blocking=True)
        cond = batch["SR"].to(device, non_blocking=True)
        loss = training_loss(net, hr, cond)
        opt_g.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip:
            torch.nn.utils.clip_grad_norm_(net.parameters(), float(grad_clip))
        opt_g.step()

        if step % 100 == 0:
            log.info("dmahr iter %d/%d loss %.5f (%.0fs)", step, iterations,
                     float(loss), time.time() - t0)
        if step % val_every == 0 or step == iterations:
            v = val_loss(net, val_loader, device)
            history.append({"iteration": step, "train_l1": float(loss), "val_l1": v})
            log.info("dmahr val iter %d: %.5f", step, v)
            if v < best:
                best = v
                torch.save(net.state_dict(), out_dir / "best_gen.pth")
        if step % ckpt_every == 0 or step == iterations:
            tmp = state_path.with_suffix(".tmp")
            torch.save(
                {"iteration": step, "model": net.state_dict(),
                 "optimizer": opt_g.state_dict(), "best_val": best,
                 "history": history},
                tmp,
            )
            tmp.replace(state_path)
        # Iteration-numbered weights at a coarser interval, so a learning curve
        # can be measured afterwards without retraining.
        snap_every = int(cfg["train"].get("snapshot_every", 5000))
        if snap_every and (step % snap_every == 0 or step == iterations):
            torch.save(net.state_dict(), out_dir / f"gen_iter{step:06d}.pth")

    torch.save(net.state_dict(), out_dir / "final_gen.pth")
    save_json({"history": history, "best_val": best, "iterations": iterations,
               "seconds": round(time.time() - t0, 1)}, out_dir / "train_log.json")
    return {"iterations": iterations, "best_val": best, "seconds": time.time() - t0}


## 6a. ArmDataset and transforms

The exact evaluation and training transforms shared by every ViT arm.


In [ ]:
"""Arm datasets over the frozen splits, with the matched ViT recipe."""


from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_transforms(cfg: dict, train: bool):
    """Eval: square resize + ImageNet normalize. Train: the recipe in vit.json."""
    size = int(cfg["image_size"])
    aug = cfg.get("augmentation", {})
    norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(
                size,
                scale=tuple(aug.get("random_resized_crop_scale", (0.75, 1.0))),
                ratio=tuple(aug.get("random_resized_crop_ratio", (3.0 / 4.0, 4.0 / 3.0))),
            ),
            transforms.RandomHorizontalFlip(float(aug.get("horizontal_flip", 0.5))),
            transforms.RandomVerticalFlip(float(aug.get("vertical_flip", 0.5))),
            transforms.RandomRotation(float(aug.get("rotation_degrees", 15))),
            transforms.ColorJitter(
                *[float(v) for v in aug.get("color_jitter", [0.15, 0.15, 0.1, 0.02])]
            ),
            transforms.ToTensor(),
            norm,
        ])
    # Eval: resize the shorter side to int(size * 1.14) = 255, then centre-crop.
    # This is the pipeline the recorded runs used; squashing to a square or
    # resizing to 256 changes predictions on these 600x450 images.
    return transforms.Compose([*eval_ops(size), transforms.ToTensor(), norm])

    return transforms.Compose([*eval_ops(size), transforms.ToTensor(), norm])


class ArmDataset(Dataset):
    """One arm of the study over one split.

    Arm A reads the original ISIC JPEGs; every processed arm (B, C, Astar,
    Creleased, ...) reads ``processed_root/<arm>/<image_id>.png``.
    """

    def __init__(self, arm: str, split: str, splits_df: pd.DataFrame,
                 raw_images_dir, processed_root, transform, image_ids=None):
        df = splits_df[splits_df["split"] == split]
        if image_ids is not None:
            df = df[df["image_id"].isin(set(image_ids))]
        self.records = df.reset_index(drop=True)
        self.arm = arm
        self.raw_images_dir = Path(raw_images_dir)
        self.processed_root = Path(processed_root)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.records)

    def _path(self, image_id: str) -> Path:
        if self.arm == "A":
            return self.raw_images_dir / f"{image_id}.jpg"
        return self.processed_root / self.arm / f"{image_id}.png"

    def __getitem__(self, idx: int) -> dict:
        r = self.records.iloc[idx]
        with Image.open(self._path(r["image_id"])) as fh:
            img = fh.convert("RGB")
            x = self.transform(img)
        return {
            "image": x,
            "label": int(CLASS_TO_IDX[r["dx"]]),
            "image_id": str(r["image_id"]),
            "group_id": str(r["group_id"]),
            "dx": str(r["dx"]),
        }


def collate(batch: list) -> dict:
    return {
        "image": torch.stack([b["image"] for b in batch]),
        "label": torch.tensor([b["label"] for b in batch], dtype=torch.long),
        "image_id": [b["image_id"] for b in batch],
        "group_id": [b["group_id"] for b in batch],
        "dx": [b["dx"] for b in batch],
    }


## 6b. ViT-B/16 model builder

ImageNet-pretrained torchvision ViT with a 7-class head.


In [ ]:
"""torchvision ViT-B/16 with a fixed seven-class head."""


import torch
from torch import nn
from torchvision import models



def build_model(pretrained: bool = True, num_classes: int = len(CLASSES)) -> nn.Module:
    weights = models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.vit_b_16(weights=weights)
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model


def save_checkpoint(
    path,
    model: nn.Module,
    optimizer,
    scheduler,
    epoch: int,
    best_f1: float,
    meta: dict,
) -> None:
    tmp = str(path) + ".tmp"
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_val_macro_f1": best_f1,
            "meta": meta,
        },
        tmp,
    )
    import os

    os.replace(tmp, path)


def load_checkpoint(path, model: nn.Module, map_location="cpu") -> dict:
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    model.load_state_dict(ckpt["model"])
    return ckpt


@torch.no_grad()
def predict_proba(model: nn.Module, loader, device: str) -> tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    probs, labels = [], []
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
            logits = model(x)
        probs.append(torch.softmax(logits.float(), dim=1).cpu())
        labels.append(batch["label"])
    return torch.cat(probs), torch.cat(labels)


## 6c. Classification metrics

Macro-F1, balanced accuracy, AUC and per-class metrics used for validation selection.


In [ ]:
"""Classification metrics with explicit handling of undefined values."""


import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)



def classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray | None = None,
) -> dict:
    labels = list(range(len(CLASSES)))
    p, r, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )
    out = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "per_class": {},
    }
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    for i, c in enumerate(CLASSES):
        tp = int(cm[i, i])
        fn = int(cm[i, :].sum() - tp)
        fp = int(cm[:, i].sum() - tp)
        tn = int(cm.sum() - tp - fn - fp)
        sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")
        specificity = tn / (tn + fp) if (tn + fp) else float("nan")
        auc, auc_note = _auc_for(y_true, y_prob, i)
        out["per_class"][c] = {
            "support": int(support[i]),
            "precision": float(p[i]),
            "recall": float(r[i]),
            "f1": float(f1[i]),
            "sensitivity": sensitivity,
            "specificity": specificity,
            "auc": auc,
            "auc_note": auc_note,
        }
    if y_prob is not None:
        out["auc_macro"] = _macro_auc(y_true, y_prob)
        out["auc_weighted"] = _weighted_auc(y_true, y_prob)
    else:
        out["auc_macro"] = float("nan")
        out["auc_weighted"] = float("nan")
    out["confusion_matrix"] = cm.tolist()
    out["confusion_matrix_normalized"] = (
        (cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)).round(6).tolist()
    )
    return out


def _auc_for(y_true: np.ndarray, y_prob: np.ndarray | None, idx: int):
    if y_prob is None:
        return float("nan"), "no_probabilities"
    binary = (y_true == idx).astype(int)
    if binary.sum() == 0 or binary.sum() == len(binary):
        return float("nan"), "single_class_present"
    return float(roc_auc_score(binary, y_prob[:, idx])), ""


def _macro_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    vals = [v for v in (_auc_for(y_true, y_prob, i)[0] for i in range(len(CLASSES)))
            if not np.isnan(v)]
    return float(np.mean(vals)) if vals else float("nan")


def _weighted_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    vals, weights = [], []
    for i in range(len(CLASSES)):
        v = _auc_for(y_true, y_prob, i)[0]
        if not np.isnan(v):
            vals.append(v)
            weights.append(int((y_true == i).sum()))
    if not vals:
        return float("nan")
    return float(np.average(vals, weights=weights))


def metrics_table(metrics_by_key: dict[tuple, dict]) -> pd.DataFrame:
    """Flatten many metric dicts into one CSV-friendly table."""
    rows = []
    for key, m in metrics_by_key.items():
        arm, seed, subset = key
        base = {
            "arm": arm,
            "seed": seed,
            "subset": subset,
            "n": m["n"],
            "accuracy": m["accuracy"],
            "balanced_accuracy": m["balanced_accuracy"],
            "macro_f1": m["macro_f1"],
            "weighted_f1": m["weighted_f1"],
            "auc_macro": m["auc_macro"],
            "auc_weighted": m["auc_weighted"],
        }
        for c in CLASSES:
            pc = m["per_class"][c]
            for k in ("support", "precision", "recall", "f1", "sensitivity",
                      "specificity", "auc"):
                base[f"{c}_{k}"] = pc[k]
            base[f"{c}_auc_note"] = pc["auc_note"]
        rows.append(base)
    return pd.DataFrame(rows)


## 6d. ViT-B/16 training loop

ImageNet-pretrained ViT with the matched recipe; resumes from last.pt.


In [ ]:
"""Matched ViT-B/16 training with resumable checkpoints and early stopping."""


import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import f1_score
from torch import nn
from torch.utils.data import DataLoader


log = setup_logging()


def class_weights(splits_df, class_order=CLASSES) -> torch.Tensor:
    """Inverse-frequency weights computed from TRAIN labels only."""
    counts = splits_df[splits_df["split"] == "train"]["dx"].value_counts()
    total = counts.sum()
    w = [total / (len(class_order) * max(counts.get(c, 0), 1)) for c in class_order]
    return torch.tensor(w, dtype=torch.float32)


def cosine_with_warmup(optimizer, warmup_epochs: int, total_epochs: int, steps_per_epoch: int):
    warmup = max(1, warmup_epochs * steps_per_epoch)
    total = max(warmup + 1, total_epochs * steps_per_epoch)

    def fn(step: int) -> float:
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


@torch.no_grad()
def evaluate(model, loader, device: str, criterion=None) -> dict:
    model.eval()
    probs, labels = [], []
    loss_sum, n = 0.0, 0
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
            logits = model(x)
        if criterion is not None:
            loss_sum += float(criterion(logits.float(), y)) * y.numel()
            n += y.numel()
        probs.append(torch.softmax(logits.float(), dim=1).cpu())
        labels.append(y.cpu())
    prob = torch.cat(probs)
    label = torch.cat(labels)
    pred = prob.argmax(dim=1)
    return {
        "prob": prob,
        "label": label,
        "pred": pred,
        "loss": loss_sum / max(n, 1),
        "macro_f1": float(f1_score(label, pred, average="macro", zero_division=0)),
        "accuracy": float((pred == label).float().mean()),
    }


def train_arm(
    arm: str,
    seed: int,
    cfg: dict,
    splits_df,
    raw_images_dir: Path,
    processed_root: Path,
    run_dir: Path,
    device: str = "cuda",
    max_epochs: int | None = None,
) -> dict:
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)

    train_ds = ArmDataset(arm, "train", splits_df, raw_images_dir, processed_root,
                          build_transforms(cfg, train=True))
    val_ds = ArmDataset(arm, "val", splits_df, raw_images_dir, processed_root,
                        build_transforms(cfg, train=False))
    bs = int(cfg["batch_size"])
    nw = int(cfg["num_workers"])
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=False,
                              num_workers=nw, pin_memory=True, collate_fn=collate,
                              persistent_workers=nw > 0)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=nw,
                            pin_memory=True, collate_fn=collate, persistent_workers=nw > 0)

    model = build_model(pretrained=True).to(device)
    weights = class_weights(splits_df).to(device)
    criterion = nn.CrossEntropyLoss(
        weight=weights, label_smoothing=float(cfg.get("label_smoothing", 0.0))
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["weight_decay"])
    )
    epochs = int(max_epochs or cfg["epochs"])
    scheduler = cosine_with_warmup(optimizer, int(cfg["warmup_epochs"]), epochs, len(train_loader))
    scaler = torch.amp.GradScaler("cuda", enabled=False)

    meta = {
        "arm": arm,
        "seed": seed,
        "config": cfg,
        "class_weights": weights.detach().cpu().tolist(),
        "class_order": list(CLASSES),
        "train_size": len(train_ds),
        "val_size": len(val_ds),
    }
    save_json(meta, run_dir / "run_meta.json")

    last_path = run_dir / "last.pt"
    best_path = run_dir / "best.pt"
    start_epoch, best_f1, bad_epochs, history = 0, -1.0, 0, []
    if last_path.exists():
        ckpt = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"] + 1
        best_f1 = ckpt["best_val_macro_f1"]
        bad_epochs = ckpt["meta"].get("bad_epochs", 0)
        history = ckpt["meta"].get("history", [])
        log.info("resumed %s/%d at epoch %d", arm, seed, start_epoch)

    patience = int(cfg["early_stopping_patience"])
    t0 = time.time()
    for epoch in range(start_epoch, epochs):
        model.train()
        run_loss, seen = 0.0, 0
        for batch in train_loader:
            x = batch["image"].to(device, non_blocking=True)
            y = batch["label"].to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
                logits = model(x)
                loss = criterion(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg["grad_clip"]))
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            run_loss += float(loss) * y.numel()
            seen += y.numel()
        val = evaluate(model, val_loader, device, criterion)
        rec = {
            "epoch": epoch,
            "train_loss": run_loss / max(seen, 1),
            "val_loss": val["loss"],
            "val_macro_f1": val["macro_f1"],
            "val_accuracy": val["accuracy"],
            "lr": scheduler.get_last_lr()[0],
            "seconds": round(time.time() - t0, 1),
        }
        history.append(rec)
        log.info("%s/%d ep %d: train %.4f val %.4f f1 %.4f", arm, seed, epoch,
                 rec["train_loss"], rec["val_loss"], rec["val_macro_f1"])

        if val["macro_f1"] > best_f1:
            best_f1 = val["macro_f1"]
            bad_epochs = 0
            save_checkpoint(best_path, model, optimizer, scheduler, epoch, best_f1, meta)
        else:
            bad_epochs += 1
        meta_epoch = {**meta, "bad_epochs": bad_epochs, "history": history}
        save_checkpoint(last_path, model, optimizer, scheduler, epoch, best_f1, meta_epoch)
        if bad_epochs >= patience:
            log.info("%s/%d early stop at epoch %d (best %.4f)", arm, seed, epoch, best_f1)
            break

    save_checkpoint(run_dir / "final.pt", model, optimizer, scheduler, epoch, best_f1,
                    {**meta, "bad_epochs": bad_epochs, "history": history})
    result = {
        "arm": arm,
        "seed": seed,
        "best_val_macro_f1": best_f1,
        "epochs_run": len(history),
        "seconds": round(time.time() - t0, 1),
        "history": history,
    }
    save_json(result, run_dir / "train_result.json")
    return result


## 6e. Model-frame geometry (eval_ops / mask_to_model_frame)

The single definition of the classifier's view: resize the shorter side to int(size*1.14), then centre-crop. Masks go through the identical transform.


In [ ]:
"""The one place that defines the model's view of an image and its lesion mask.

The ViT never sees the stored photo: it sees the photo resized so the shorter side is
``int(size * 1.14)`` and then centre-cropped to ``size`` x ``size``. Saliency maps live in that
cropped frame, so a lesion mask has to be carried through the identical resize and crop before
it is compared with a map. The classifier's evaluation transform (``data.dataset``) and every
explanation script call the functions below, so the two cannot drift apart.

Stored image sizes differ by arm AND by image: the original and DullRazor arms keep the 600 x 450
photo; in the A*, C, Creleased and Cv2 arms the hair-present images are stored at 128 x 128 (an
aspect-ratio squash) while hair-absent images pass through unchanged at 600 x 450 (see
`scripts/02_hair_pipeline.py`). The geometry therefore reads each image's stored size from the file.
Masks are stored at 600 x 450 and are squashed to that size first.
"""


import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

EVAL_SCALE = 1.14
FULL_SIZE = (600, 450)
SQUASHED_SIZE = (128, 128)
ARM_STORED_SIZES: dict[str, frozenset[tuple[int, int]]] = {
    "A": frozenset({FULL_SIZE}), "B": frozenset({FULL_SIZE}),
    "Astar": frozenset({FULL_SIZE, SQUASHED_SIZE}), "C": frozenset({FULL_SIZE, SQUASHED_SIZE}),
    "Creleased": frozenset({FULL_SIZE, SQUASHED_SIZE}), "Cv2": frozenset({FULL_SIZE, SQUASHED_SIZE}),
}
FRAME_VERSION = "model_frame_v2"


def eval_resize_size(size: int = 224) -> int:
    return int(size * EVAL_SCALE)


def eval_ops(size: int = 224, interpolation=InterpolationMode.BILINEAR) -> list:
    """The resize and centre-crop of the classifier's evaluation transform."""
    return [transforms.Resize(eval_resize_size(size), interpolation=interpolation),
            transforms.CenterCrop(size)]


def to_model_frame(img: Image.Image, size: int = 224, is_mask: bool = False) -> Image.Image:
    """Carry a PIL image (or mask, with nearest-neighbour resampling) into the model's frame."""
    mode = InterpolationMode.NEAREST if is_mask else InterpolationMode.BILINEAR
    for op in eval_ops(size, mode):
        img = op(img)
    return img


def mask_to_model_frame(mask: Image.Image, stored_size_wh: tuple[int, int] | None,
                        size: int = 224) -> np.ndarray:
    """Binary {0,1} lesion mask in the model's frame, for an arm whose images are stored at
    ``stored_size_wh``. ``None`` means the mask is already at the stored size."""
    m = mask.convert("L")
    if stored_size_wh is not None and m.size != tuple(stored_size_wh):
        m = m.resize(tuple(int(v) for v in stored_size_wh), Image.Resampling.NEAREST)
    m = to_model_frame(m, size, is_mask=True)
    return (np.asarray(m) > 127).astype(np.uint8)


def shift_zero_fill(a, dx: int, dy: int):
    """Translate an (..., H, W) numpy array or tensor by (dx, dy) pixels, filling with zeros."""
    is_t = isinstance(a, torch.Tensor)
    out = torch.zeros_like(a) if is_t else np.zeros_like(a)
    h, w = a.shape[-2], a.shape[-1]
    ys, yd = (slice(0, h - dy), slice(dy, h)) if dy >= 0 else (slice(-dy, h), slice(0, h + dy))
    xs, xd = (slice(0, w - dx), slice(dx, w)) if dx >= 0 else (slice(-dx, w), slice(0, w + dx))
    if abs(dx) < w and abs(dy) < h:
        out[..., yd, xd] = a[..., ys, xs]
    return out


## 6f. Grad-CAM, attention rollout and localisation metrics

Grad-CAM on the 14x14 patch-embedding grid and identity-blended attention rollout, plus overlap, pointing game and chance baselines.


In [ ]:
"""Explainability maps for torchvision ViT-B/16.

Grad-CAM is taken on `conv_proj`, whose output is the (B, 768, 14, 14) patch
grid. Attention rollout multiplies identity-blended attention matrices captured
from the real self-attention modules.
"""


import contextlib

import numpy as np
import torch
import torch.nn.functional as F


def _patch_grid(model) -> int:
    size = model.image_size if isinstance(model.image_size, int) else model.image_size[0]
    return size // model.patch_size


class GradCAM:
    """Grad-CAM on the patch-embedding grid (conv_proj)."""

    def __init__(self, model, layer=None):
        self.model = model
        self.layer = layer if layer is not None else model.conv_proj
        self.activations: torch.Tensor | None = None
        self.gradients: torch.Tensor | None = None
        self._handle = None

    def __enter__(self):
        def fwd_hook(_module, _inp, out):
            self.activations = out
            if out.requires_grad:
                out.register_hook(self._save_grad)

        self._handle = self.layer.register_forward_hook(fwd_hook)
        return self

    def __exit__(self, *exc):
        if self._handle is not None:
            self._handle.remove()
            self._handle = None
        return False

    def _save_grad(self, grad):
        self.gradients = grad

    def __call__(self, x: torch.Tensor, class_idx: int | None = None) -> np.ndarray:
        """Return a normalized (grid, grid) CAM for one image tensor (1,3,H,W)."""
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        # Grad-CAM needs a gradient path back to the patch grid, so the input
        # must require grad even though the model is in eval mode.
        x = x.detach().clone().requires_grad_(True)
        with torch.enable_grad():
            logits = self.model(x)
            if class_idx is None:
                class_idx = int(logits.argmax(dim=1))
            score = logits[0, class_idx]
            score.backward(retain_graph=False)
        if self.gradients is None:
            raise RuntimeError("Grad-CAM captured no gradients on conv_proj")
        act = self.activations.detach()[0]           # (768, g, g)
        grad = self.gradients.detach()[0]            # (768, g, g)
        weights = grad.mean(dim=(1, 2), keepdim=True)
        cam = F.relu((weights * act).sum(dim=0))
        cam = cam - cam.min()
        denom = cam.max()
        if float(denom) > 0:
            cam = cam / denom
        return cam.cpu().numpy()


@contextlib.contextmanager
def capture_attention(model):
    """Capture averaged attention weights from every block's self_attn module."""
    captured: list[torch.Tensor] = []
    originals = []

    for block in model.encoder.layers:
        # torchvision names this module `self_attention`; timm-style code uses
        # `self_attn`. Accept either so the hook does not depend on the fork.
        attn = getattr(block, "self_attention", None) or getattr(block, "self_attn", None)
        if attn is None:
            raise AttributeError("encoder block exposes no self-attention module")
        originals.append((attn, attn.forward))

        def make(original_forward, store):
            def forward(query, key, value, *args, **kwargs):
                kwargs["need_weights"] = True
                kwargs["average_attn_weights"] = True
                out, weights = original_forward(query, key, value, *args, **kwargs)
                store.append(weights.detach())
                # torchvision unpacks `x, _ = self_attention(...)`; keep that shape.
                return out, None

            return forward

        attn.forward = make(attn.forward, captured)

    try:
        yield captured
    finally:
        for module, original in originals:
            module.forward = original


@torch.no_grad()
def attention_rollout(model, x: torch.Tensor, discard_ratio: float = 0.9) -> np.ndarray:
    """Identity-blended attention rollout, returned as a (grid, grid) map."""
    model.eval()
    with capture_attention(model) as captured:
        model(x)
    if not captured:
        raise RuntimeError("no attention weights captured")

    tokens = captured[0].shape[-1]
    result = torch.eye(tokens)
    for attn in captured:
        a = attn[0].float().cpu()
        if discard_ratio > 0:
            flat = a.flatten(1)
            k = int(flat.shape[1] * discard_ratio)
            if k > 0:
                idx = flat.topk(k, dim=1, largest=False).indices
                flat.scatter_(1, idx, 0.0)
                a = flat.view_as(a)
        a = a / a.sum(dim=-1, keepdim=True).clamp_min(1e-9)
        blended = 0.5 * a + 0.5 * torch.eye(tokens)
        result = blended @ result

    cls_attention = result[0, 1:]
    grid = _patch_grid(model)
    mask = cls_attention.reshape(grid, grid).numpy()
    mask = mask - mask.min()
    denom = mask.max()
    if denom > 0:
        mask = mask / denom
    return mask


def upsample_map(cam: np.ndarray, size: int = 224) -> np.ndarray:
    t = torch.from_numpy(cam).float()[None, None]
    return F.interpolate(t, size=(size, size), mode="bilinear",
                         align_corners=False)[0, 0].numpy()


def top_fraction_mask(saliency: np.ndarray, fraction: float = 0.2) -> np.ndarray:
    flat = saliency.ravel()
    k = max(1, int(round(flat.size * fraction)))
    idx = np.argpartition(-flat, k - 1)[:k]
    out = np.zeros(flat.size, dtype=bool)
    out[idx] = True
    return out.reshape(saliency.shape)


def lesion_overlap(saliency: np.ndarray, mask: np.ndarray, fraction: float = 0.2) -> float:
    """Overlap (mask recall): the share of the lesion covered by the top-20% most salient pixels."""
    top = top_fraction_mask(saliency, fraction)
    lesion_area = float(mask.sum())
    if lesion_area == 0:
        return float("nan")
    return float((top & mask.astype(bool)).sum() / lesion_area)


def salient_in_lesion(saliency: np.ndarray, mask: np.ndarray, fraction: float = 0.2) -> float:
    """Precision: the share of the top-20% most salient pixels that fall inside the lesion."""
    if mask.sum() == 0:
        return float("nan")
    top = top_fraction_mask(saliency, fraction)
    return float((top & mask.astype(bool)).sum() / top.sum())


def top_region_centroid_distance(saliency: np.ndarray, mask: np.ndarray, fraction: float = 0.2) -> float:
    """Centroid distance: between the centre of the top-20% region and the lesion centre, in image widths."""
    if mask.sum() == 0:
        return float("nan")
    top = top_fraction_mask(saliency, fraction)
    ty, tx = np.nonzero(top)
    ly, lx = np.nonzero(mask.astype(bool))
    d = np.hypot(tx.mean() - lx.mean(), ty.mean() - ly.mean())
    return float(d / saliency.shape[1])


def pointing_game(saliency: np.ndarray, mask: np.ndarray, tolerance: int = 0) -> float:
    """Pointing: 1 if the single most salient pixel falls inside the lesion, else 0."""
    m = mask.astype(bool)
    if tolerance > 0:
        k = 2 * tolerance + 1
        m = (
            F.max_pool2d(
                torch.from_numpy(m.astype(np.float32))[None, None], k, stride=1, padding=tolerance
            )[0, 0].numpy()
            > 0
        )
    if m.sum() == 0:
        return float("nan")
    idx = int(np.argmax(saliency))
    r, c = divmod(idx, saliency.shape[1])
    return float(bool(m[r, c]))


def uniform_chance_overlap(mask: np.ndarray, fraction: float = 0.2, seed: int = 0,
                           n: int = 1) -> float:
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        sal = rng.random(mask.shape)
        vals.append(lesion_overlap(sal, mask, fraction))
    return float(np.nanmean(vals))


def gaussian_chance_map(size: int = 224, sigma_frac: float = 0.25) -> np.ndarray:
    yy, xx = np.mgrid[0:size, 0:size]
    c = (size - 1) / 2.0
    sigma = size * sigma_frac
    g = np.exp(-(((xx - c) ** 2 + (yy - c) ** 2) / (2 * sigma**2)))
    return g / g.max()


def overlay_heatmap(image: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    import matplotlib.cm as cm

    heat = cm.jet(cam)[..., :3]
    base = image.astype(np.float32) / 255.0
    return np.clip((1 - alpha) * base + alpha * heat, 0, 1).astype(np.float32)


## 6g. Lesion-cluster bootstrap, Wilcoxon and Holm

Paired resampling at lesion-group level, group-aggregated Wilcoxon and Holm step-down correction.


In [ ]:
"""Paired cluster bootstrap and Holm correction.

All resampling is done at the lesion-group level, and the same resample is
shared across arms and seeds so comparisons stay paired.
"""


import numpy as np
import pandas as pd


def holm(pvalues: list[float]) -> list[float]:
    """Holm step-down adjustment. NaN inputs are ignored in the family."""
    idx = [i for i, p in enumerate(pvalues) if p is not None and not np.isnan(p)]
    m = len(idx)
    out = [float("nan")] * len(pvalues)
    if m == 0:
        return out
    ordered = sorted(idx, key=lambda i: pvalues[i])
    running = 0.0
    for rank, i in enumerate(ordered):
        adj = (m - rank) * float(pvalues[i])
        running = max(running, adj)
        out[i] = min(1.0, running)
    return out


def paired_cluster_bootstrap(
    values_a: np.ndarray,
    values_b: np.ndarray,
    groups: np.ndarray,
    statistic=None,
    replicates: int = 10000,
    seed: int = 20260101,
    alpha: float = 0.05,
) -> dict:
    """Bootstrap the paired difference a - b, resampling lesion groups.

    `values_a`/`values_b` hold one observation per row (same row alignment).
    For image-level metric tables, aggregate to one row per group first.
    """
    a = np.asarray(values_a, dtype=float)
    b = np.asarray(values_b, dtype=float)
    g = np.asarray(groups)
    ok = np.isfinite(a) & np.isfinite(b)
    a, b, g = a[ok], b[ok], g[ok]
    if len(a) == 0:
        return {"n": 0, "note": "no finite paired observations"}

    stat = statistic or (lambda x: float(np.mean(x)))
    uniq = np.unique(g)
    index_by_group = {gu: np.nonzero(g == gu)[0] for gu in uniq}
    observed = stat(a) - stat(b)
    rng = np.random.default_rng(seed)
    diffs = np.empty(replicates, dtype=float)
    for r in range(replicates):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        rows = np.concatenate([index_by_group[gu] for gu in pick])
        diffs[r] = stat(a[rows]) - stat(b[rows])
    lo, hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    p_two = 2 * min(float((diffs <= 0).mean()), float((diffs >= 0).mean()))
    sd = float(np.std(diffs, ddof=1)) if replicates > 1 else float("nan")
    return {
        "n": int(len(a)),
        "n_groups": int(len(uniq)),
        "stat_a": float(stat(a)),
        "stat_b": float(stat(b)),
        "difference": float(observed),
        "ci_low": float(lo),
        "ci_high": float(hi),
        "p_raw": float(min(1.0, p_two)),
        "effect_size_dz": float(observed / sd) if sd and sd > 0 else float("nan"),
        "replicates": int(replicates),
        "seed": int(seed),
        "alpha": alpha,
    }


def cohens_dz(differences: np.ndarray) -> float:
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    if len(d) < 2 or d.std(ddof=1) == 0:
        return float("nan")
    return float(d.mean() / d.std(ddof=1))


def wilcoxon_group_paired(df: pd.DataFrame, value: str, arm_col: str,
                          group_col: str = "group_id") -> dict:
    """Group-aggregated Wilcoxon signed-rank between the two arms in `df`."""
    from scipy.stats import wilcoxon

    wide = df.pivot_table(index=group_col, columns=arm_col, values=value, aggfunc="mean")
    wide = wide.dropna()
    if wide.shape[1] != 2 or len(wide) < 6:
        return {"n_groups": int(len(wide)), "p_raw": float("nan"),
                "note": "insufficient paired groups"}
    arms = list(wide.columns)
    try:
        res = wilcoxon(wide[arms[0]], wide[arms[1]])
        p = float(res.pvalue)
        stat = float(res.statistic)
    except Exception as exc:  # noqa: BLE001
        return {"n_groups": int(len(wide)), "p_raw": float("nan"), "note": repr(exc)}
    return {
        "test": "wilcoxon_signed_rank_group_aggregated",
        "arm_a": arms[0],
        "arm_b": arms[1],
        "n_groups": int(len(wide)),
        "statistic": stat,
        "p_raw": p,
        "median_difference": float((wide[arms[0]] - wide[arms[1]]).median()),
    }


def wilcoxon_per_group(long_df: pd.DataFrame, arm_a: str, arm_b: str,
                       value_col: str = "value",
                       group_col: str = "group_id") -> dict:
    """Wilcoxon signed-rank on lesion-group-level paired values.

    `long_df` carries one row per (arm, group). Collapsing to a single value per
    lesion first keeps the many images of one lesion from being treated as
    independent replicates.
    """
    from scipy.stats import wilcoxon

    out = {"wilcoxon_n_groups": 0, "wilcoxon_statistic": float("nan"),
           "wilcoxon_p_raw": float("nan")}
    if long_df is None or long_df.empty:
        return out
    sub = long_df[long_df["arm"].isin([arm_a, arm_b])]
    if sub.empty:
        return out
    wide = sub.pivot_table(index=group_col, columns="arm",
                           values=value_col, aggfunc="mean").dropna()
    out["wilcoxon_n_groups"] = int(len(wide))
    if wide.shape[1] != 2 or len(wide) < 6:
        return out
    a = wide[arm_a].to_numpy(dtype=float)
    b = wide[arm_b].to_numpy(dtype=float)
    if np.allclose(a - b, 0.0):
        out["wilcoxon_statistic"] = 0.0
        out["wilcoxon_p_raw"] = 1.0
        return out
    try:
        res = wilcoxon(a, b)
    except ValueError:
        return out
    out["wilcoxon_statistic"] = float(res.statistic)
    out["wilcoxon_p_raw"] = float(res.pvalue)
    return out


## 6h. Composite image metrics

Helpers for the image-quality stage.


In [ ]:
"""Per-item metrics for hair removal on composites with known clean targets.

All arrays are at the same resolution (the diffusion model's 128 px). `alpha` is the soft
strand coverage of the composite, `lesion` a boolean lesion mask. Lesion damage is measured
only on lesion pixels the hair did not touch, where any change from the input is damage.
"""


import cv2
import numpy as np

CHANGE_LEVELS = 10
HAIR_ALPHA = 0.02
MIN_LESION_PIXELS = 50


def item_metrics(out: np.ndarray, target: np.ndarray, comp: np.ndarray, alpha: np.ndarray,
                 lesion: np.ndarray) -> dict:
    out, target, comp = (np.asarray(x, np.float64) for x in (out, target, comp))
    mse = float(np.mean((out - target) ** 2))
    psnr = float(10.0 * np.log10(255.0**2 / mse)) if mse > 0 else float("inf")
    w = alpha[..., None].astype(np.float64)
    denom = float(w.sum() * 3 + 1e-9)
    e_floor = float((np.abs(comp - target) * w).sum() / denom)
    e_out = float((np.abs(out - target) * w).sum() / denom)
    removal = 1.0 - e_out / max(e_floor, 1e-9)

    hair_region = cv2.dilate((alpha > HAIR_ALPHA).astype(np.uint8),
                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))) > 0
    clear_lesion = lesion & ~hair_region
    clear_skin = ~lesion & ~hair_region
    changed = np.abs(out - comp).max(axis=2) > CHANGE_LEVELS
    err = np.abs(out - target).mean(axis=2)
    nan = float("nan")
    return {
        "psnr": psnr,
        "hair_removal": removal,
        "lesion_pixels_clear_of_hair": int(clear_lesion.sum()),
        "lesion_mae": float(err[clear_lesion].mean()) if clear_lesion.sum() >= MIN_LESION_PIXELS else nan,
        "lesion_changed": float(changed[clear_lesion].mean()) if clear_lesion.sum() >= MIN_LESION_PIXELS else nan,
        "skin_mae": float(err[clear_skin].mean()) if clear_skin.sum() >= MIN_LESION_PIXELS else nan,
        "skin_changed": float(changed[clear_skin].mean()) if clear_skin.sum() >= MIN_LESION_PIXELS else nan,
    }


def bootstrap_mean_ci(x: np.ndarray, reps: int = 2000, seed: int = 0) -> tuple[float, float, float]:
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    means = x[rng.integers(0, x.size, (reps, x.size))].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(x.mean()), float(lo), float(hi)


## 7. Phase 1 — download ISIC 212, HAM10000 metadata and lesion masks

Public data, no account. Re-running skips files already cached on Drive.


In [ ]:
stats = download_collection(212, isic_images_dir(), raw_dir() / "isic_212" / "records.jsonl", workers=16)

def fetch(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return {"path": str(dest), "cached": True}
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + ".part")
        with tmp.open("wb") as fh:
            for chunk in r.iter_content(1 << 20): fh.write(chunk)
        tmp.replace(dest)
    return {"path": str(dest), "cached": False}

meta = fetch(DATAVERSE_ACCESS.format(fid=METADATA_FILE_ID), ham_root() / "HAM10000_metadata.tab")
mzip = fetch(DATAVERSE_ACCESS.format(fid=MASKS_FILE_ID), ham_root() / "HAM10000_segmentations_lesion_tschandl.zip")
extracted = extract_masks(Path(mzip["path"]), ham_masks_dir())
save_json({"isic": stats, "metadata": meta, "masks": mzip, "extracted": extracted},
          artifact_dir("logs") / "download_report.json")
print("ISIC:", stats)


## 8. Phase 2 — audit and freeze the lesion-grouped split

Counts are recomputed from files; confirmed near-duplicate links are merged into lesion groups before the split. The leakage check must pass.


In [ ]:
protocol = load_config("protocol.json")
protocol_hash = config_hash_file("protocol.json")
records_path = raw_dir() / "isic_212" / "records.jsonl"
metadata = read_metadata(ham_root() / "HAM10000_metadata.tab")
comparison = expected_comparison(metadata)

out_manifest = manifest_dir() / "data_manifest.csv"
n_records = len([ln for ln in records_path.read_text().splitlines() if ln.strip()])
man = None
if out_manifest.exists():
    cached = pd.read_csv(out_manifest)
    if len(cached) == n_records and cached["sha256"].notna().all() and cached["dhash"].notna().all():
        man = cached
if man is None:
    man, _, _ = build_manifest(records_path, isic_images_dir(), metadata, manifest_dir())

c0 = man[(man["cohort"] == "C0") & (man["decode_status"] == "ok")].copy()
c0["dx"] = c0["dx"].astype(str); c0["lesion_id"] = c0["lesion_id"].astype(str)
cand_path = manifest_dir() / "near_duplicates.csv"
if cand_path.exists():
    candidates = pd.read_csv(cand_path)
else:
    hashes = c0["dhash"].astype(np.uint64).to_numpy()
    candidates = pd.DataFrame(hamming_pairs(hashes, 6, c0["image_id"].tolist()),
                              columns=["id_a", "id_b", "distance"])
    candidates.to_csv(cand_path, index=False)
conf_path = manifest_dir() / "near_duplicates_confirmed.csv"
if conf_path.exists():
    confirmed = pd.read_csv(conf_path)
else:
    confirmed = confirm_near_duplicates(candidates, man, max_mean_abs_diff=4.0, min_correlation=0.985)
    if len(confirmed): confirmed.to_csv(conf_path, index=False)
kept = confirmed[confirmed["confirmed"]] if len(confirmed) else confirmed
grouped = build_groups(c0, kept if len(kept) else candidates.iloc[0:0])
summary = freeze_splits(grouped, seed=int(protocol["split_seed"]),
                        fractions=protocol["split_fractions"],
                        out_dir=manifest_dir(), protocol_hash=protocol_hash)
split_df = pd.read_csv(manifest_dir() / "splits.csv")
verification = verify_splits(split_df)
x1 = build_x1_manifest(ham_masks_dir(), split_df, man, manifest_dir() / "mask_manifest_x1.csv")
save_json({"collection_total": int(len(man)), "c0_decoded": int(len(c0)),
           "frozen_splits": summary, "split_verification": verification,
           "near_duplicates_confirmed": int(len(kept))},
          artifact_dir("logs") / "data_audit.json")
print("split hash:", summary["splits_sha256"], "| verification:", verification)
assert verification["ok"] and comparison["all_match"]


## 9. Phase 3 — hair screening, Arm B, A* and synthetic pairs


In [ ]:
SYNTH_SEED = 7000
SYNTH_LIMITS = {"train": 3000, "val": 800, "test": 186}

def read_rgb(path):
    with Image.open(path) as im: return np.array(im.convert("RGB"))
def write_png(arr, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(arr).save(path, compress_level=3)

screen_cfg = load_config("hair_screen.json")
dull_cfg = load_config("dullrazor.json")
splits = pd.read_csv(manifest_dir() / "splits.csv")
if not screen_cfg.get("frozen"):
    tune_rows = splits[splits["split"].isin(["train", "val"])]
    sample = tune_rows.sample(n=min(400, len(tune_rows)), random_state=42)
    pairs = []
    for _, r in sample.iterrows():
        img = read_rgb(isic_images_dir() / f"{r['image_id']}.jpg")
        if detect_hair(img, {**screen_cfg, "coverage_threshold": 10.0})[1]["coverage"] > 0.01: continue
        hairy, _, _ = render_hair(Image.fromarray(img), r["image_id"], SYNTH_SEED + 1)
        pairs += [(img, 0), (np.array(hairy), 1)]
    best = tune_coverage_threshold(pairs, screen_cfg)
    screen_cfg = {**screen_cfg, "coverage_threshold": best["coverage_threshold"], "frozen": True}
    screen_cfg["config_hash"] = config_hash({k: v for k, v in screen_cfg.items() if k != "config_hash"})
    dull_cfg = {**dull_cfg, "frozen": True}
    dull_cfg["config_hash"] = config_hash({k: v for k, v in dull_cfg.items() if k != "config_hash"})
    print("froze screen:", best)

def process_one(row):
    iid = row["image_id"]
    out = {"image_id": iid, "split": row["split"], "dx": row["dx"],
           "group_id": row["group_id"], "screen_config_hash": screen_cfg["config_hash"]}
    try: img = read_rgb(isic_images_dir() / f"{iid}.jpg")
    except Exception as exc:
        out.update({"status": "fail_decode", "error": repr(exc), "hair_present": None, "hair_coverage": None}); return out
    mask, stats = detect_hair(img, screen_cfg)
    out.update({"width": img.shape[1], "height": img.shape[0],
                "hair_coverage": round(stats["coverage"], 6),
                "hair_present": stats["hair_present"], "status": "ok"})
    try:
        write_png(mask * 255, processed_dir("hair_masks") / f"{iid}.png")
        if stats["hair_present"]:
            write_png(apply_dullrazor(img, mask, dull_cfg), processed_dir("B") / f"{iid}.png")
            write_png(np.array(Image.fromarray(img).resize((128, 128), Image.Resampling.BICUBIC)),
                      processed_dir("Astar") / f"{iid}.png")
        else:
            write_png(img, processed_dir("B") / f"{iid}.png")
            write_png(img, processed_dir("Astar") / f"{iid}.png")
    except Exception as exc: out.update({"status": "fail_write", "error": repr(exc)})
    return out

with ThreadPoolExecutor(max_workers=min(48, (os.cpu_count() or 4) * 2)) as ex:
    rows = list(ex.map(process_one, [r for _, r in splits.iterrows()]))
df = pd.DataFrame(rows)
df.to_csv(manifest_dir() / "hair_screening.csv", index=False)
print("screened", len(df), "hair present", int(df["hair_present"].sum()))


In [ ]:
def synth_one(row, split):
    iid = row["image_id"]
    img = read_rgb(isic_images_dir() / f"{iid}.jpg")
    hairy, hmask, params = render_hair(Image.fromarray(img), iid, SYNTH_SEED)
    base = processed_dir("synth") / split
    write_png(np.array(hairy), base / "hairy" / f"{iid}.png")
    write_png(img, base / "reference" / f"{iid}.png")
    write_png(np.array(hmask), base / "hair_mask" / f"{iid}.png")
    return {"pair_id": f"synth_{split}_{iid}", "image_id": iid, "split": split,
            "dx": row["dx"], "group_id": row["group_id"],
            "hairy_path": str(base / "hairy" / f"{iid}.png"),
            "reference_path": str(base / "reference" / f"{iid}.png"),
            "hair_mask_path": str(base / "hair_mask" / f"{iid}.png"),
            "seed": SYNTH_SEED, "n_hairs": params["n_hairs"],
            "blur_sigma": params["blur"], "source": "synthetic"}

synth_rows, absent = [], df[~df["hair_present"]]
for split, limit in SYNTH_LIMITS.items():
    sub = [r for _, r in absent[absent["split"] == split].head(limit).iterrows()]
    with ThreadPoolExecutor(max_workers=min(32, (os.cpu_count() or 4) * 2)) as ex:
        synth_rows += list(ex.map(lambda r, s=split: synth_one(r, s), sub))
pd.DataFrame(synth_rows).to_csv(manifest_dir() / "synthetic_pairs.csv", index=False)
print("synthetic pairs:", len(synth_rows))


## 10. Phase 4 — harvest pasted real strands (train/val only)


In [ ]:
hcfg = load_config("hair_screen.json")
scr = pd.read_csv(manifest_dir() / "hair_screening.csv")
scr = scr[scr["split"].isin(["train", "val"]) & (scr["hair_coverage"] >= HAIR_MIN_COVERAGE)]
assert not (scr["split"] == "test").any()
layers_dir = data_root() / "hair_layers"; layers_dir.mkdir(parents=True, exist_ok=True)
harvest_rows = []
for iid in sorted(scr["image_id"]):
    m, info = thin_hair_mask(read_rgb(isic_images_dir() / f"{iid}.jpg"), hcfg)
    if info["hair_pixels"] > 0:
        write_png((m * 255).astype(np.uint8), layers_dir / f"{iid}.png")
    harvest_rows.append({"image_id": iid, **info})
pd.DataFrame(harvest_rows).to_csv(layers_dir / "summary.csv", index=False)
print("hair layers:", len(list(layers_dir.glob("*.png"))))


## 11. Phase 5 — train the repaired base diffusion generator (50k)


In [ ]:
cfg_dm = load_config("dmahr.json")
pairs = pd.read_csv(manifest_dir() / "synthetic_pairs.csv")
train_pairs, val_pairs = pairs[pairs["split"] == "train"], pairs[pairs["split"] == "val"]
assert set(train_pairs["group_id"]).isdisjoint(set(val_pairs["group_id"]))
result_base = train(cfg_dm, train_pairs, val_pairs, artifact_dir("checkpoints", "dmahr"),
                    max_iterations=DIFF_BASE_ITERS)
save_json(result_base, artifact_dir("logs") / "dmahr_train.json")
base_ckpts = sorted((artifact_dir("checkpoints", "dmahr")).glob("gen_iter*.pth"),
                    key=lambda p: int(p.stem.split("_")[-1]))
base_ckpt = base_ckpts[-1]
print("base checkpoint:", base_ckpt)


## 12. Phase 6 — fine-tune for 20k iterations on pasted real strands


In [ ]:
train_ds, val_ds = build_datasets(manifest_dir() / "hair_screening.csv", isic_images_dir(),
                                  layers_dir, size=int(cfg_dm["native_resolution"]))
result_ft = train(cfg_dm, None, None, artifact_dir("checkpoints", "dmahr_v2"),
                  init_from=base_ckpt, max_iterations=DIFF_FINETUNE_ITERS,
                  train_ds=train_ds, val_ds=val_ds, num_workers=8)
save_json(result_ft, artifact_dir("logs") / "dmahr_v2_train.json")
ft_ckpt = artifact_dir("checkpoints", "dmahr_v2") / "final_gen.pth"
assert ft_ckpt.exists()
print("fine-tuned checkpoint:", ft_ckpt)


## 13. Phase 7 — generate Cv2 images (128 px, 50 strided DDIM steps, t0=600, FP32)


In [ ]:
net = load_model(cfg_dm, ft_ckpt, device="cuda")

def sample_shard(shard, n_shards, pairs_split=None):
    if pairs_split:
        p = pd.read_csv(manifest_dir() / "synthetic_pairs.csv")
        p = p[p["split"] == pairs_split].sort_values("image_id").iloc[shard::n_shards]
        todo = pd.DataFrame({"image_id": p["image_id"].values,
                             "source_path": [str(_resolve_manifest_path(q)) for q in p["hairy_path"].values]})
        out_dir = processed_dir("synth") / pairs_split / "cleaned_Cv2"
    else:
        s = pd.read_csv(manifest_dir() / "hair_screening.csv")
        s = s[s["hair_present"]].sort_values("image_id").reset_index(drop=True).iloc[shard::n_shards]
        todo = pd.DataFrame({"image_id": s["image_id"].values,
                             "source_path": [str(isic_images_dir() / f"{i}.jpg") for i in s["image_id"]]})
        out_dir = processed_dir("Cv2")
    out_dir.mkdir(parents=True, exist_ok=True)
    todo = todo[~todo["image_id"].map(lambda i: (out_dir / f"{i}.png").exists())]
    bs = int(cfg_dm["sampling"]["batch_size"]); done = 0
    for start in range(0, len(todo), bs):
        chunk = todo.iloc[start:start + bs]
        images = [Image.open(p).convert("RGB") for p in chunk["source_path"]]
        seed = int(hashlib.sha256(f"{shard}:{start}".encode()).hexdigest()[:8], 16)
        outs, stats = clean_batch(net, images, cfg_dm, device="cuda", seed=seed,
                                  num_timesteps=CV2_STEPS, use_bf16=False, respaced=True,
                                  start_t=CV2_START_T)
        assert stats["finite"]
        for iid, out in zip(chunk["image_id"], outs):
            tmp = out_dir / f"{iid}.png.tmp"
            out.save(tmp, format="PNG"); tmp.replace(out_dir / f"{iid}.png"); done += 1
    return done

for shard in range(CV2_SHARDS):
    print("shard", shard, "processed", sample_shard(shard, CV2_SHARDS))
print("held-out pairs:", sample_shard(0, 1, pairs_split="test"))
passthrough = pd.read_csv(manifest_dir() / "hair_screening.csv")
for iid in passthrough[~passthrough["hair_present"]]["image_id"]:
    dest = processed_dir("Cv2") / f"{iid}.png"
    if not dest.exists(): shutil.copy2(isic_images_dir() / f"{iid}.jpg", dest)
print("Cv2 complete:", len(list(processed_dir("Cv2").glob("*.png"))))


## 14. Phase 8 — train the ViT-B/16 classifiers (last training phase)


In [ ]:
cfg_vit = load_config("vit.json")
splits_df = pd.read_csv(manifest_dir() / "splits.csv")
for arm in ARMS:
    for seed in SEEDS:
        res = train_arm(arm=arm, seed=seed, cfg=cfg_vit, splits_df=splits_df,
                        raw_images_dir=isic_images_dir(), processed_root=data_root() / "processed",
                        run_dir=artifact_dir("checkpoints", "vit", f"{arm}_seed{seed}"),
                        device="cuda")
        print(arm, seed, "best val macro-F1", round(res["best_val_macro_f1"], 4), flush=True)


## 16. Phase 9 — H1: image quality on paired hairless references

Two explicit geometry policies, never mixed: (a) both images at 128 px; (b) the 128 px diffusion/classical output upsampled to the native reference size. Fixed data range [0,1], RGB, SSIM window 7. The public DERMAHAIR pairs are **NOT VERIFIED** in this notebook: the released checkpoint weights and that archive are not embedded, so they are not scored here.


In [ ]:
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

def score_pair(pred, ref, policy="128px"):
    if policy == "128px":
        p = np.asarray(Image.fromarray(pred).resize((128, 128), Image.Resampling.BICUBIC), dtype=np.float64) / 255.0
        r = np.asarray(Image.fromarray(ref).resize((128, 128), Image.Resampling.BICUBIC), dtype=np.float64) / 255.0
    elif policy == "native":
        p = np.asarray(Image.fromarray(pred).resize(Image.fromarray(ref).size, Image.Resampling.BICUBIC), dtype=np.float64) / 255.0
        r = np.asarray(ref, dtype=np.float64) / 255.0
    else:
        raise ValueError(policy)
    return {"psnr": float(peak_signal_noise_ratio(r, p, data_range=1.0)),
            "ssim": float(structural_similarity(r, p, data_range=1.0, channel_axis=2, win_size=7))}

pairs = pd.read_csv(manifest_dir() / "synthetic_pairs.csv")
pairs = pairs[pairs["split"] == "test"].reset_index(drop=True).head(300)
rows_iq = []
for r in pairs.itertuples(index=False):
    hairy = np.asarray(Image.open(str(_resolve_manifest_path(r.hairy_path))).convert("RGB"))
    ref = np.asarray(Image.open(str(_resolve_manifest_path(r.reference_path))).convert("RGB"))
    mask, _ = detect_hair(hairy, screen_cfg)
    b_out = apply_dullrazor(hairy, mask, dull_cfg)
    c_path = processed_dir("synth") / "test" / "cleaned_Cv2" / f"{r.image_id}.png"
    if not c_path.exists():
        continue
    c_out = np.asarray(Image.open(c_path).convert("RGB"))
    row = {"pair_id": r.pair_id, "image_id": r.image_id, "dx": r.dx, "group_id": r.group_id}
    for arm, arr in (("hairy_input", hairy), ("B", b_out), ("Cv2", c_out)):
        for pol in ("128px", "native"):
            m = score_pair(arr, ref, pol)
            row[f"{arm}_{pol}_psnr"], row[f"{arm}_{pol}_ssim"] = m["psnr"], m["ssim"]
    rows_iq.append(row)
iq = pd.DataFrame(rows_iq)
iq.to_csv(artifact_dir("logs") / "paired_psnr_ssim_test.csv", index=False)
print(iq[[f"{a}_{p}_psnr" for a in ("hairy_input","B","Cv2") for p in ("128px","native")]].mean().round(3))


## 17. Phase 10 — H2: classification, per-image predictions and full test metrics

Every frozen checkpoint is scored once on the locked test set. Outputs include per-image probabilities, full and hair-present subsets, per-class precision/recall/F1/specificity/AUC, weighted F1 and raw/normalised confusion matrices.


In [ ]:
from torch.utils.data import DataLoader
from dm_ahr_vit.vit.model import build_model, load_checkpoint

def predict(model, loader, device):
    model.eval(); rows = []
    with torch.no_grad():
        for batch in loader:
            x = batch["image"].to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
                logits = model(x)
            prob = torch.softmax(logits.float(), dim=1).cpu().numpy()
            for i, iid in enumerate(batch["image_id"]):
                row = {"image_id": iid, "group_id": batch["group_id"][i],
                       "label": batch["dx"][i], "pred": CLASSES[int(prob[i].argmax())],
                       "prob_max": float(prob[i].max())}
                for j, c in enumerate(CLASSES): row[f"prob_{c}"] = float(prob[i, j])
                rows.append(row)
    return pd.DataFrame(rows)

def extra_metrics(df):
    y = np.array([CLASSES.index(v) for v in df["label"]])
    p = np.array([CLASSES.index(v) for v in df["pred"]])
    prob = df[[f"prob_{c}" for c in CLASSES]].to_numpy()
    base = classification_metrics(y, p, prob)
    cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
    for t, q in zip(y, p): cm[t, q] += 1
    out = dict(base)
    for k, c in enumerate(CLASSES):
        tp, fp = cm[k, k], cm[:, k].sum() - cm[k, k]
        fn, tn = cm[k, :].sum() - cm[k, k], cm.sum() - tp - fp - fn
        out[f"specificity_{c}"] = float(tn / (tn + fp)) if tn + fp else float("nan")
        out[f"support_{c}"] = int(cm[k, :].sum())
    out["weighted_f1"] = float(sum(out[f"support_{c}"] * out["f1"][c] for c in CLASSES) / max(len(y), 1))
    out["confusion"] = cm.tolist()
    out["confusion_normalised"] = (cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)).round(4).tolist()
    return out

splits_hash = sha256_file(manifest_dir() / "splits.csv")
hairy_ids = set(pd.read_csv(manifest_dir() / "hair_screening.csv").query("hair_present")["image_id"])
all_preds, summary_rows = [], []
for arm in ARMS:
    for seed in SEEDS:
        run_dir = artifact_dir("checkpoints", "vit", f"{arm}_seed{seed}")
        ckpt_path = run_dir / "best.pt"
        if not ckpt_path.exists():
            print("NOT VERIFIED: missing checkpoint", ckpt_path); continue
        model = build_model(pretrained=True).to("cuda")
        ckpt = load_checkpoint(ckpt_path, model, map_location="cuda")
        tfm = build_transforms(cfg_vit, train=False)
        for subset, ids in (("full", None), ("hair_present", hairy_ids)):
            ds = ArmDataset(arm, "test", splits_df, isic_images_dir(), data_root() / "processed", tfm, image_ids=ids)
            if len(ds) == 0: continue
            loader = DataLoader(ds, batch_size=int(cfg_vit["batch_size"]), shuffle=False,
                                num_workers=0, collate_fn=collate)
            df = predict(model, loader, "cuda")
            df.insert(0, "subset", subset); df.insert(0, "seed", seed); df.insert(0, "arm", arm)
            all_preds.append(df)
            m = extra_metrics(df)
            summary_rows.append({"arm": arm, "seed": seed, "subset": subset, **{k: v for k, v in m.items() if k != "per_class"}})
        del model; torch.cuda.empty_cache()
pred_df = pd.concat(all_preds, ignore_index=True)
pred_df.to_csv(artifact_dir("predictions") / "per_image_predictions.csv", index=False)
pd.DataFrame(summary_rows).to_csv(artifact_dir("logs") / "classification_metrics_by_seed_v2.csv", index=False)
pd.DataFrame([r for r in summary_rows if r["subset"] == "hair_present"]).to_csv(
    artifact_dir("logs") / "hair_present_subset_metrics.csv", index=False)
print("scored", len(pred_df), "prediction rows; splits hash", splits_hash[:16])


## 18. Phase 11 — H3: explainability vs lesion masks

Grad-CAM (patch grid) and attention rollout are upsampled to 224x224 and compared with the released X1 lesion masks in the classifier's own frame, with uniform, centred-Gaussian and random baselines. Overlays are saved under `artifacts/panels/`.


In [ ]:
masks = pd.read_csv(manifest_dir() / "mask_manifest_x1.csv")
def mask_path(row):
    p = Path(str(row["mask_path"]))
    return ham_masks_dir() / p.name

xai_rows = []
for arm in ARMS:
    ckpt_path = artifact_dir("checkpoints", "vit", f"{arm}_seed{SEEDS[0]}") / "best.pt"
    if not ckpt_path.exists(): continue
    model = build_model(pretrained=True).to("cuda")
    load_checkpoint(ckpt_path, model, map_location="cuda")
    ds = ArmDataset(arm, "test", splits_df, isic_images_dir(), data_root() / "processed",
                    build_transforms(cfg_vit, train=False))
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate)
    gc = GradCAM(model)
    with gc:
        for batch in loader:
            iid = batch["image_id"][0]
            mrow = masks[masks["image_id"] == iid]
            if mrow.empty or not bool(mrow.iloc[0]["has_mask"]): continue
            mimg = Image.open(mask_path(mrow.iloc[0])).convert("L")
            lesion = mask_to_model_frame(mimg, mimg.size, size=224)
            x = batch["image"].to("cuda")
            cam = upsample_map(gc(x), 224)
            roll = upsample_map(attention_rollout(model, x), 224)
            gauss = gaussian_chance_map(224)
            rng = np.random.default_rng(0)
            rand = rng.random((224, 224))
            row = {"arm": arm, "image_id": iid, "group_id": batch["group_id"][0]}
            for name, sal in (("gradcam", cam), ("rollout", roll), ("gaussian", gauss), ("random", rand)):
                row[f"{name}_overlap"] = lesion_overlap(sal, lesion, 0.2)
                row[f"{name}_pointing"] = pointing_game(sal, lesion, 0)
            xai_rows.append(row)
    del model; torch.cuda.empty_cache()
xdf = pd.DataFrame(xai_rows)
xdf.to_csv(artifact_dir("logs") / "xai_localization_test.csv", index=False)
print(xdf.groupby("arm")[[c for c in xdf.columns if c.endswith(("_overlap","_pointing"))]].mean().round(3))


## 19. Phase 12 — statistics: lesion-cluster bootstrap, Wilcoxon and Holm

Two Holm families are reported: the original protocol family (A-B, A-C, B-C) and the current planned family (A-B only; others exploratory). Bootstrap p-values below the resolution floor are printed as `< 1/(replicates+1)`, never as 0.0.


In [ ]:
REPS, BSEED = 10000, 20261113
summary = pd.read_csv(artifact_dir("logs") / "classification_metrics_by_seed_v2.csv")
full = summary[summary["subset"] == "full"]
wide = full.pivot_table(index=["group_id", "seed"], columns="arm", values="macro_f1")
# group-level values: mean over seeds per arm per lesion group
gdf = full.groupby(["group_id", "arm"])["macro_f1"].mean().unstack().dropna()
def fam(cols, names):
    rows, pv = [], []
    for (a, b), nm in zip(cols, names):
        r = paired_cluster_bootstrap(gdf[a].to_numpy(), gdf[b].to_numpy(),
                                     gdf.index.to_numpy(), replicates=REPS, seed=BSEED)
        rows.append({"comparison": nm, "family": "protocol" if nm in ("A-B","A-C","B-C") else "current",
                     **r})
        pv.append(r["p_raw"])
    adj = holm(pv)
    for r, p in zip(rows, adj): r["p_holm"] = p
    return rows
protocol_rows = fam([("A","B"),("A","C"),("B","C")], ["A-B","A-C","B-C"])
current_rows = fam([("A","B")], ["A-B"])
stats_df = pd.DataFrame(protocol_rows + current_rows)
stats_df["wilcoxon_p_raw"] = [wilcoxon_group_paired(full.rename(columns={"macro_f1":"value"}), "value", "arm").get("p_raw")
                              if c == "A-B" else float("nan") for c in stats_df["comparison"]]
stats_df.to_csv(artifact_dir("logs") / "statistical_comparisons_v2.csv", index=False)
print(stats_df[["comparison","family","difference","ci_low","ci_high","p_raw","p_holm"]].to_string(index=False))


## 20. Phase 13 — reproducibility notes and deliverables

Everything below is written from the run just performed. Nothing is typed by hand. Unavailable evidence is marked NOT VERIFIED with the reason.


In [ ]:
notes = [
    "# Reproducibility notes", "",
    "- Splits SHA-256: " + splits_hash,
    "- Protocol Holm family: A-B, A-C, B-C (10,000 lesion-group bootstrap replicates, seed " + str(BSEED) + ").",
    "- Current planned family: A-B only; all other comparisons exploratory.",
    "- Config hashes frozen in the CONFIGS dictionary at the top of this notebook.",
    "- Public DERMAHAIR PSNR/SSIM: NOT VERIFIED in this notebook (released weights and archive not embedded).",
    "- Lama et al. independent hair masks for screening QA: NOT VERIFIED in this notebook (not embedded; used only in the separate external validation).",
    "- Expert-traced masks: NOT APPLICABLE. X1 masks are single-rater segmentations, FCN-initialised then corrected by a dermatologist (Tschandl et al.), not expert-traced.",
    "- Lost checkpoints: the ~56.3k continuation generator and the first defective generator are absent from the workspace; 'every number can be regenerated' is therefore not claimed.",
    "- Blinded human review: prepared as pending, no ratings filled in.",
    "",
    "## Files written by this notebook", "",
    "| Artifact | Rows / count |", "|---|---:|",
    f"| artifacts/logs/data_audit.json | splits hash {splits_hash[:12]}… |",
    f"| artifacts/logs/screening_audit.csv | {len(df)} images screened |",
    f"| artifacts/logs/paired_psnr_ssim_test.csv | {len(iq)} pairs x 2 policies |",
    f"| artifacts/predictions/per_image_predictions.csv | {len(pred_df)} rows |",
    f"| artifacts/logs/classification_metrics_by_seed_v2.csv | {len(summary_rows)} arm/seed/subset rows |",
    f"| artifacts/logs/xai_localization_test.csv | {len(xdf)} explained images |",
    f"| artifacts/logs/statistical_comparisons_v2.csv | {len(stats_df)} comparisons |",
]
(artifact_dir("reports") / "reproducibility_notes.md").write_text("\n".join(notes) + "\n")
print("\n".join(notes))


## 15. Models produced — that is the whole job


In [ ]:
ck = ARTIFACT_ROOT / "checkpoints"
diffusion = sorted((ck / "dmahr").glob("gen_iter*.pth")) + sorted((ck / "dmahr_v2").glob("*.pth"))
vit = sorted(ck.glob("vit/*/best.pt"))
print("diffusion generators:", len(diffusion))
for p in diffusion: print(" ", p.relative_to(ck), f"{p.stat().st_size/1e6:.1f} MB")
print("ViT checkpoints:", len(vit))
for p in vit: print(" ", p.relative_to(ck), f"{p.stat().st_size/1e6:.1f} MB")
print("\nAll models under:", ck)
